In [1]:
# 1 Config


from __future__ import annotations

import os

# Thread policy must run BEFORE importing numpy/pandas (BLAS/OpenMP init).
def _resolve_threads(requested: int) -> int:
    cpu = os.cpu_count() or 1
    if isinstance(requested, int) and requested > 0:
        return max(1, min(requested, cpu))
    return max(1, cpu)



# 1.1 User knobs

SEED: int = 42

# -1 => use all cores on this machine, or set a positive cap.
THREADS_REQUESTED: int = -1
THREADS: int = _resolve_threads(THREADS_REQUESTED)

#CPU usage
LIMIT_BLAS_THREADS: bool = True
if LIMIT_BLAS_THREADS:
    os.environ["OMP_NUM_THREADS"] = str(THREADS)
    os.environ["MKL_NUM_THREADS"] = str(THREADS)
    os.environ["OPENBLAS_NUM_THREADS"] = str(THREADS)
    os.environ["NUMEXPR_NUM_THREADS"] = str(THREADS)

# Python-level determinism knobs
os.environ["PYTHONHASHSEED"] = str(SEED)

import sys
import json
import time
import random
import hashlib
import platform
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Any

random.seed(SEED)

import numpy as np
import pandas as pd

np.random.seed(SEED)



# 1.2 Paths

# Default layout:
#   ./shift_ml_2026_train.csv
#   ./shift_ml_2026_test.csv
PROJECT_DIR = Path.cwd()
DATA_DIR = Path(os.getenv("SHIFT_DATA_DIR", PROJECT_DIR))
OUTPUTS_DIR = Path(os.getenv("SHIFT_OUTPUTS_DIR", PROJECT_DIR / "outputs"))
CACHE_DIR = Path(os.getenv("SHIFT_CACHE_DIR", PROJECT_DIR / "cache"))


# 1.3 Config objects

@dataclass(frozen=True)
class CFGCore:
    seed: int = SEED
    threads: int = THREADS

    id_col: str = "id"
    target_col: str = "итоговый_статус_займа"

    train_path: Path = DATA_DIR / "shift_ml_2026_train.csv"
    test_path: Path = DATA_DIR / "shift_ml_2026_test.csv"

    out_dir: Path = OUTPUTS_DIR
    cache_dir: Path = CACHE_DIR


@dataclass(frozen=True)
class CFGRun:
    run_name_prefix: str = "run"
    verbose: bool = True

    save_environment: bool = True
    save_pip_freeze: bool = True
    save_fingerprints: bool = True

    limit_blas_threads: bool = LIMIT_BLAS_THREADS


@dataclass(frozen=True)
class CFGPrep:
    # strings
    nan_like: tuple[str, ...] = ("", "NAN", "NONE", "NULL", "NA", "N/A")

    # binary-like strings
    convert_binary_strings: bool = True
    unknown_as_na: bool = True
    unknown_tokens: tuple[str, ...] = (
        "НЕИЗВЕСТНО", "НЕТ ДАННЫХ", "НЕ УКАЗАНО", "ОТСУТСТВУЕТ", "UNKNOWN",
        "N/A", "NA", "NULL", "NONE", ""
    )

    # dates
    parse_dates: bool = True
    date_cols_explicit: tuple[str, ...] = ("дата_первого_займа",)
    date_prefixes: tuple[str, ...] = ("дата_",)
    date_min_parse_rate: float = 0.01
    drop_raw_dates: bool = True  # keep simple; can switch later

    # profession tail bucketing
    use_prof_rare: bool = True
    prof_col: str = "профессия_заемщика"
    rare_token: str = "__RARE__"
    missing_token: str = "__MISSING__"

    # tail control: keep ~X% of rows in frequent professions
    prof_target_keep_share: float = 0.95
    prof_min_count_floor: int = 10
    prof_max_count_cap: int = 1000
    # Optional hard threshold (disabled by default; see Step 3)
    # prof_min_count_fixed: int | None = 10

    # profession superclass (auto, token-based)
    prof_super_max_n: int = 50   
    prof_super_max_n_cap: int = 500 
    prof_super_min_df: int = 50
    prof_super_max_df_share: float = 0.30
    prof_super_other_token: str = "__OTHER__"
    prof_super_generic_token: str = "__GENERIC__"
    prof_super_tail_other_token: str = "__OTHER_TAIL__"

    use_profession_step4 = True
    drop_raw_profession = True
    drop_norm_profession = True
    prof_superclass_col = "профессия_superclass"
    prof_seniority_col  = "профессия_seniority"
    prof_domain_col     = "профессия_domain"

    # optional: parallel implementation
    prof_super_n_jobs: int = -1         # -1 => core.threads

    # viability / stability
    near_const_thresh: float | None = None  # e.g. 0.999
    near_const_max_nunique: int = 20

    # known columns
    rating_col: str = "рейтинг"
    doprating_col: str = "допрейтинг"


core = CFGCore()
run_cfg = CFGRun()
prep = CFGPrep()


# 1.4 Helpers

def ensure_dirs(*paths: Path) -> None:
    for p in paths:
        p.mkdir(parents=True, exist_ok=True)


def require_file(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"{label} not found: {path}\n"
            f"Tip: place files into {DATA_DIR} or set SHIFT_DATA_DIR."
        )
    if path.stat().st_size <= 0:
        raise ValueError(f"{label} is empty: {path}")


def file_fingerprint(path: Path) -> Dict[str, Any]:
    st = path.stat()
    return {
        "name": path.name,
        "path": str(path),
        "size_bytes": int(st.st_size),
        "mtime_ns": int(st.st_mtime_ns),
    }


def safe_import_version(pkg_name: str) -> str:
    try:
        mod = __import__(pkg_name)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return "not_installed"


def collect_environment(core: CFGCore, run_cfg: CFGRun) -> Dict[str, Any]:
    return {
        "os": platform.platform(),
        "python": platform.python_version(),
        "python_executable": sys.executable,
        "cwd": str(Path.cwd()),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": safe_import_version("scipy"),
        "sklearn": safe_import_version("sklearn"),
        "catboost": safe_import_version("catboost"),
        "pyarrow": safe_import_version("pyarrow"),
        "matplotlib": safe_import_version("matplotlib"),
        "threads_resolved": int(core.threads),
        "threads_requested": int(THREADS_REQUESTED),
        "limit_blas_threads": bool(run_cfg.limit_blas_threads),
        "env_vars_threads": {
            "OMP_NUM_THREADS": os.getenv("OMP_NUM_THREADS"),
            "MKL_NUM_THREADS": os.getenv("MKL_NUM_THREADS"),
            "OPENBLAS_NUM_THREADS": os.getenv("OPENBLAS_NUM_THREADS"),
            "NUMEXPR_NUM_THREADS": os.getenv("NUMEXPR_NUM_THREADS"),
        },
    }


def pip_freeze_text() -> str:
    try:
        return subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    except Exception as e:
        return f"# pip freeze failed: {repr(e)}\n"


def now_run_id_utc() -> str:
    ts = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
    suffix = hashlib.blake2b(f"{time.time_ns()}".encode(), digest_size=3).hexdigest()
    return f"{ts}_{suffix}"


def write_json(path: Path, obj: Any) -> None:
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")



# 1.5 Init: dirs, checks, run artifacts

ensure_dirs(core.out_dir, core.cache_dir)

require_file(core.train_path, "TRAIN CSV")
require_file(core.test_path, "TEST  CSV")

RUN_ID = now_run_id_utc()
RUN_DIR = core.out_dir / "runs" / f"{run_cfg.run_name_prefix}_{RUN_ID}"
ensure_dirs(RUN_DIR)

# Save configs (convert Paths to strings)
cfg_dump = {
    "CFGCore": {
        **asdict(core),
        "train_path": str(core.train_path),
        "test_path": str(core.test_path),
        "out_dir": str(core.out_dir),
        "cache_dir": str(core.cache_dir),
    },
    "CFGRun": asdict(run_cfg),
    "CFGPrep": asdict(prep),
}
write_json(RUN_DIR / "config.json", cfg_dump)

if run_cfg.save_environment:
    write_json(RUN_DIR / "environment.json", collect_environment(core, run_cfg))

if run_cfg.save_pip_freeze:
    (RUN_DIR / "requirements_freeze.txt").write_text(pip_freeze_text(), encoding="utf-8")

if run_cfg.save_fingerprints:
    fp = {
        "train": file_fingerprint(core.train_path),
        "test": file_fingerprint(core.test_path),
    }
    write_json(RUN_DIR / "data_fingerprints.json", fp)

if run_cfg.verbose:
    print("=== Init OK ===")
    print("RUN_DIR:", RUN_DIR)
    print(f"seed={core.seed} | threads={core.threads} (requested={THREADS_REQUESTED})")
    print("train:", core.train_path)
    print("test :", core.test_path)
    print("saved:", (RUN_DIR / "config.json").name,
          "|", ("environment.json" if run_cfg.save_environment else "no env"),
          "|", ("requirements_freeze.txt" if run_cfg.save_pip_freeze else "no freeze"),
          "|", ("data_fingerprints.json" if run_cfg.save_fingerprints else "no fp"))


=== Init OK ===
RUN_DIR: C:\Users\aa\Documents\SHIFT_ML\SHIFT_ML_2026_COMPETITION\outputs\runs\run_20260209_092450_05dcdc
seed=42 | threads=8 (requested=-1)
train: C:\Users\aa\Documents\SHIFT_ML\SHIFT_ML_2026_COMPETITION\shift_ml_2026_train.csv
test : C:\Users\aa\Documents\SHIFT_ML\SHIFT_ML_2026_COMPETITION\shift_ml_2026_test.csv
saved: config.json | environment.json | requirements_freeze.txt | data_fingerprints.json


In [2]:
# 2 Load data & sanity checks
# Read raw train/test, validate schema, and dump small reports into RUN_DIR.
# Cache is optional; EDA is optional and runs on a sampled fraction with caps.

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# 2.1 Cache policy

@dataclass(frozen=True)
class CachePolicy:
    use_cache: bool = True
    save_cache: bool = True
    engine: str = "pyarrow"
    compression: str = "zstd"
    strict_meta_match: bool = True  # rebuild cache if CSV changed


CACHE = CachePolicy(use_cache=True, save_cache=True, strict_meta_match=True)


def _cache_paths(cache_dir: Path) -> Tuple[Path, Path, Path, Path]:
    train_pq = cache_dir / "train_raw.parquet"
    test_pq = cache_dir / "test_raw.parquet"
    train_meta = cache_dir / "train_raw.meta.json"
    test_meta = cache_dir / "test_raw.meta.json"
    return train_pq, test_pq, train_meta, test_meta


def _read_json(path: Path) -> Optional[Dict[str, Any]]:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def _write_json(path: Path, obj: Any) -> None:
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def _fingerprint_for_cache(path: Path) -> Dict[str, Any]:
    st = path.stat()
    return {"path": str(path), "size_bytes": int(st.st_size), "mtime_ns": int(st.st_mtime_ns)}


def _cache_is_valid(csv_path: Path, parquet_path: Path, meta_path: Path) -> Tuple[bool, str]:
    if not parquet_path.exists():
        return False, "parquet_missing"
    if not meta_path.exists():
        return False, "meta_missing"
    meta = _read_json(meta_path)
    if not isinstance(meta, dict) or "csv_fingerprint" not in meta:
        return False, "meta_bad_format"
    if meta["csv_fingerprint"] != _fingerprint_for_cache(csv_path):
        return False, "fingerprint_mismatch"
    return True, "ok"


def _save_cache(
    df: pd.DataFrame,
    parquet_path: Path,
    meta_path: Path,
    csv_path: Path,
    cache: CachePolicy,
) -> None:
    df.to_parquet(parquet_path, index=False, engine=cache.engine, compression=cache.compression)
    meta = {
        "csv_fingerprint": _fingerprint_for_cache(csv_path),
        "parquet_path": str(parquet_path),
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
    }
    _write_json(meta_path, meta)



# 2.2 Load raw data

def load_raw_data(
    *,
    core: CFGCore,
    run_dir: Path,
    cache: CachePolicy,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_pq, test_pq, train_meta, test_meta = _cache_paths(core.cache_dir)

    # Keep id stable; the rest stays raw for Step 3.
    dtype_id = {core.id_col: "Int64"}

    def _read_csv_train() -> pd.DataFrame:
        return pd.read_csv(core.train_path, low_memory=False, dtype=dtype_id)

    def _read_csv_test() -> pd.DataFrame:
        return pd.read_csv(core.test_path, low_memory=False, dtype=dtype_id)

    loaded_from_cache = False
    cache_diag: Dict[str, Any] = {}

    if cache.use_cache:
        if cache.strict_meta_match:
            ok_tr, why_tr = _cache_is_valid(core.train_path, train_pq, train_meta)
            ok_te, why_te = _cache_is_valid(core.test_path, test_pq, test_meta)
        else:
            ok_tr, why_tr = (train_pq.exists(), "ok" if train_pq.exists() else "parquet_missing")
            ok_te, why_te = (test_pq.exists(), "ok" if test_pq.exists() else "parquet_missing")

        cache_diag = {
            "train_cache_ok": bool(ok_tr),
            "test_cache_ok": bool(ok_te),
            "train_cache_reason": why_tr,
            "test_cache_reason": why_te,
            "train_parquet": str(train_pq),
            "test_parquet": str(test_pq),
        }

        if ok_tr and ok_te:
            train = pd.read_parquet(train_pq)
            test = pd.read_parquet(test_pq)
            loaded_from_cache = True
        else:
            train = _read_csv_train()
            test = _read_csv_test()
    else:
        train = _read_csv_train()
        test = _read_csv_test()

    # Parquet may restore slightly different dtype; normalize again.
    train[core.id_col] = train[core.id_col].astype("Int64")
    test[core.id_col] = test[core.id_col].astype("Int64")

    # Cache is optional
    if (not loaded_from_cache) and cache.save_cache:
        try:
            _save_cache(train, train_pq, train_meta, core.train_path, cache)
            _save_cache(test, test_pq, test_meta, core.test_path, cache)
        except Exception as e:
            (run_dir / "cache_warning.txt").write_text(
                f"Parquet cache save skipped: {repr(e)}\n", encoding="utf-8"
            )

    _write_json(
        run_dir / "step2_load.json",
        {
            "loaded_from_cache": loaded_from_cache,
            "train_shape": [int(train.shape[0]), int(train.shape[1])],
            "test_shape": [int(test.shape[0]), int(test.shape[1])],
            "cache_policy": {
                "use_cache": cache.use_cache,
                "save_cache": cache.save_cache,
                "strict_meta_match": cache.strict_meta_match,
                "engine": cache.engine,
                "compression": cache.compression,
            },
            "cache_diag": cache_diag,
        },
    )

    return train, test



# 2.3 Sanity checks

def sanity_checks(
    *,
    train: pd.DataFrame,
    test: pd.DataFrame,
    core: CFGCore,
) -> Dict[str, Any]:
    ID = core.id_col
    TGT = core.target_col

    if ID not in train.columns or ID not in test.columns:
        raise ValueError(f"Missing id_col='{ID}' in train/test")
    if train[ID].isna().any() or test[ID].isna().any():
        raise ValueError(
            f"ID contains NA: train_na={int(train[ID].isna().sum())}, test_na={int(test[ID].isna().sum())}"
        )

    if TGT not in train.columns:
        raise ValueError(f"Missing target_col='{TGT}' in train")
    if TGT in test.columns:
        raise ValueError("Target must not be present in test")

    dup_train = int(train[ID].duplicated().sum())
    dup_test = int(test[ID].duplicated().sum())
    if dup_train > 0 or dup_test > 0:
        raise ValueError(f"Duplicate ids found: train={dup_train}, test={dup_test}")

    # Target must be 0/1 (numeric).
    y = pd.to_numeric(train[TGT], errors="coerce")
    if y.isna().any():
        raise ValueError(f"Target contains non-numeric/NA values: {int(y.isna().sum())} rows")
    y = y.astype(int)
    uniq = sorted(set(y.unique().tolist()))
    if uniq not in ([0, 1], [0], [1]):
        raise ValueError(f"Unexpected target values (expected 0/1): {uniq}")

    # Test must contain all train features (extras are fine).
    train_feats = [c for c in train.columns if c not in (ID, TGT)]
    test_feats = [c for c in test.columns if c != ID]

    missing_in_test = sorted(set(train_feats) - set(test_feats))
    extra_in_test = sorted(set(test_feats) - set(train_feats))

    if missing_in_test:
        raise ValueError(f"Columns present in train but missing in test (examples): {missing_in_test[:20]}")

    # Raw "dead cols" (Step 3 will drop anyway, but it's useful to log now).
    train_all_missing = [c for c in train_feats if train[c].isna().all()]
    test_all_missing = [c for c in train_feats if test[c].isna().all()]

    vc = y.value_counts(dropna=False).sort_index()
    pos_rate = float((y == 1).mean())

    return {
        "train_shape": [int(train.shape[0]), int(train.shape[1])],
        "test_shape": [int(test.shape[0]), int(test.shape[1])],
        "id_dtype_train": str(train[ID].dtype),
        "id_dtype_test": str(test[ID].dtype),
        "n_features_train": int(len(train_feats)),
        "features_master": train_feats,  # contract for Step 3
        "extra_in_test": extra_in_test,
        "train_all_missing_cols": train_all_missing,
        "test_all_missing_cols": test_all_missing,
        "target_counts": {str(k): int(v) for k, v in vc.to_dict().items()},
        "target_pos_rate": pos_rate,
    }


def save_sanity_report(report: Dict[str, Any], run_dir: Path) -> None:
    _write_json(run_dir / "step2_sanity.json", report)
    _write_json(run_dir / "step2_features_master.json", {"features_master": report["features_master"]})


# 2.4 Optional: cache-friendly column stats for Step 3

def _is_string_like(s: pd.Series) -> bool:
    dt = s.dtype
    return pd.api.types.is_object_dtype(dt) or pd.api.types.is_string_dtype(dt)


def compute_column_stats(
    *,
    train: pd.DataFrame,
    test: pd.DataFrame,
    core: CFGCore,
    features_master: List[str],
) -> Dict[str, Any]:
    """
    Minimal stats that help Step 3 make safe decisions without re-reading CSVs.
    """
    stats: Dict[str, Any] = {"columns": {}, "n_train": int(len(train)), "n_test": int(len(test))}
    for c in features_master:
        tr = train[c]
        te = test[c]

        tr_miss = float(tr.isna().mean())
        te_miss = float(te.isna().mean())

        # Avoid expensive nunique on huge objects unless needed.
        if _is_string_like(tr):
            tr_nuniq = int(tr.nunique(dropna=True))
            te_nuniq = int(te.nunique(dropna=True))
            kind = "string"
        else:
            tr_nuniq = int(tr.nunique(dropna=True))
            te_nuniq = int(te.nunique(dropna=True))
            kind = "numeric_or_other"

        stats["columns"][c] = {
            "kind": kind,
            "train_missing": tr_miss,
            "test_missing": te_miss,
            "train_nunique_nonmiss": tr_nuniq,
            "test_nunique_nonmiss": te_nuniq,
        }
    return stats


def save_column_stats(stats: Dict[str, Any], run_dir: Path) -> None:
    _write_json(run_dir / "step2_column_stats.json", stats)


# 2.5 Run Step 2

train_raw, test_raw = load_raw_data(core=core, run_dir=RUN_DIR, cache=CACHE)

sanity = sanity_checks(train=train_raw, test=test_raw, core=core)
save_sanity_report(sanity, RUN_DIR)

# Small stats used later by Step 3 (optional but useful).
col_stats = compute_column_stats(
    train=train_raw,
    test=test_raw,
    core=core,
    features_master=sanity["features_master"],
)
save_column_stats(col_stats, RUN_DIR)

if run_cfg.verbose:
    print("=== Step 2 OK ===")
    print("train:", tuple(sanity["train_shape"]), "| test:", tuple(sanity["test_shape"]))
    print("n_features(train):", sanity["n_features_train"])
    print("target_pos_rate:", f"{sanity['target_pos_rate']:.4f}")
    if sanity["train_all_missing_cols"] or sanity["test_all_missing_cols"]:
        print("all-missing cols:", f"train={len(sanity['train_all_missing_cols'])}, test={len(sanity['test_all_missing_cols'])}")
    if sanity["extra_in_test"]:
        print("extra_in_test (ignored later):", sanity["extra_in_test"][:10], "...")



# 2.6 Optional EDA (sample only)

DO_EDA = False

if DO_EDA:
    import matplotlib.pyplot as plt

    ID = core.id_col
    TGT = core.target_col
    feat_cols: List[str] = sanity["features_master"]

    # Sample by fraction, with caps.
    EDA_SAMPLE_FRAC = 0.20
    EDA_MIN_N = 50_000
    EDA_MAX_N = 300_000

    N = len(train_raw)
    n = int(round(N * EDA_SAMPLE_FRAC))
    n = max(1, min(N, n))
    n = max(min(N, EDA_MIN_N), min(n, EDA_MAX_N))

    samp = train_raw.sample(n=n, random_state=core.seed)

    miss_all = samp[feat_cols].isna().mean().sort_values(ascending=False)
    miss_path = RUN_DIR / "eda_missing_all.csv"
    miss_all.to_frame("missing_share").to_csv(miss_path, index=True, encoding="utf-8")

    print(f"\n[EDA] sample_n={n} ({EDA_SAMPLE_FRAC:.0%} with caps {EDA_MIN_N}..{EDA_MAX_N})")
    print(f"[EDA] missingness saved to: {miss_path.name}")

    miss_row = samp[feat_cols].isna().mean(axis=1)

    plt.figure()
    plt.hist(miss_row.to_numpy(), bins=50)
    plt.title("Row missingness (sample)")
    plt.xlabel("Missing share")
    plt.ylabel("Count")
    plt.show()

    plt.figure()
    samp[TGT].value_counts().sort_index().plot(kind="bar")
    plt.title("Target distribution (sample)")
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.show()


=== Step 2 OK ===
train: (1210779, 109) | test: (134531, 108)
n_features(train): 107
target_pos_rate: 0.1996
all-missing cols: train=1, test=2


In [3]:
# 3 Preprocess (model-agnostic)
# Clean strings, add a few generic features, drop unusable cols, write artifacts.

from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd



# IO

def _write_json(path: Path, obj: Any) -> None:
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def _read_json(path: Path) -> Dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


# Strings

def _infer_string_cols(df: pd.DataFrame, exclude: set[str]) -> List[str]:
    out: List[str] = []
    for c in df.columns:
        if c in exclude:
            continue
        dt = df[c].dtype
        if pd.api.types.is_object_dtype(dt) or pd.api.types.is_string_dtype(dt):
            out.append(c)
    return out


def _normalize_strings_inplace(
    df: pd.DataFrame,
    cols: Sequence[str],
    *,
    nan_like_upper: set[str],
    collapse_spaces: bool = True,
) -> None:
    for c in cols:
        s = df[c].astype("string").str.strip()
        if collapse_spaces:
            s = s.str.replace(r"\s+", " ", regex=True)
        up = s.str.upper()
        df[c] = s.where(~up.isin(nan_like_upper), other=pd.NA)


# Binary-like conversion

def _convert_binary_like(
    train: pd.DataFrame,
    test: pd.DataFrame,
    cols: Sequence[str],
    *,
    true_set: set[str],
    false_set: set[str],
    unknown_set: set[str],
    require_both_classes: bool = True,
) -> List[str]:
    converted: List[str] = []
    allowed = true_set | false_set | unknown_set

    def _map(s: pd.Series) -> pd.Series:
        x = s.astype("string").str.strip().str.upper()
        out = pd.Series(pd.NA, index=s.index, dtype="Int8")
        out.loc[x.isin(true_set)] = 1
        out.loc[x.isin(false_set)] = 0
        if unknown_set:
            out.loc[x.isin(unknown_set)] = pd.NA
        return out.astype("Int8")

    for c in cols:
        if c not in train.columns or c not in test.columns:
            continue
        if test[c].isna().all():
            continue

        vals = pd.concat([train[c], test[c]], axis=0).dropna()
        if vals.empty:
            continue

        uniq = set(vals.astype("string").str.strip().str.upper().unique())
        if not uniq.issubset(allowed):
            continue

        has_true = bool(uniq & true_set)
        has_false = bool(uniq & false_set)
        if require_both_classes and not (has_true and has_false):
            continue

        train[c] = _map(train[c])
        test[c] = _map(test[c])
        converted.append(c)

    return converted


# Dates

def _parse_dates_expand(
    train: pd.DataFrame,
    test: pd.DataFrame,
    cols: Sequence[str],
    *,
    drop_raw: bool,
    min_parse_rate: float,
    formats: Optional[Sequence[str]] = None,
) -> Tuple[List[str], Dict[str, Any]]:
    parsed: List[str] = []
    stats: Dict[str, Any] = {}

    if formats is None:
        formats = (
            "%m-%Y",
            "%m.%Y",
            "%m/%Y",
            "%Y-%m-%d",
            "%d.%m.%Y",
            "%d-%m-%Y",
            "%d/%m/%Y",
        )

    def _parse_multi(s: pd.Series) -> Tuple[pd.Series, str]:
        x = s.astype("string").str.strip().str.replace(r"[./]", "-", regex=True)
        out = pd.Series(pd.NaT, index=x.index, dtype="datetime64[ns]")
        used = "none"

        for fmt in formats:
            dt = pd.to_datetime(x, format=fmt, errors="coerce")
            if used == "none" and dt.notna().any():
                used = fmt
            if out.isna().any():
                out = out.fillna(dt)
            if not out.isna().any():
                break

        return out, used

    for col in cols:
        if col not in train.columns or col not in test.columns:
            continue

        tr_dt, tr_fmt = _parse_multi(train[col])
        te_dt, te_fmt = _parse_multi(test[col])

        tr_rate = float(tr_dt.notna().mean())
        te_rate = float(te_dt.notna().mean())
        if tr_rate < min_parse_rate and te_rate < min_parse_rate:
            continue

        parsed.append(col)
        stats[col] = {
            "train_parse_rate": tr_rate,
            "test_parse_rate": te_rate,
            "train_format_hit": tr_fmt,
            "test_format_hit": te_fmt,
        }

        train[f"{col}_year"] = tr_dt.dt.year.astype("Int16")
        train[f"{col}_month"] = tr_dt.dt.month.astype("Int8")
        train[f"{col}_ym"] = (
            tr_dt.dt.year.astype("Int16").astype("string")
            + "-"
            + tr_dt.dt.month.astype("Int8").astype("string").str.zfill(2)
        ).astype("string")

        test[f"{col}_year"] = te_dt.dt.year.astype("Int16")
        test[f"{col}_month"] = te_dt.dt.month.astype("Int8")
        test[f"{col}_ym"] = (
            te_dt.dt.year.astype("Int16").astype("string")
            + "-"
            + te_dt.dt.month.astype("Int8").astype("string").str.zfill(2)
        ).astype("string")

        if drop_raw:
            train.drop(columns=[col], inplace=True, errors="ignore")
            test.drop(columns=[col], inplace=True, errors="ignore")

    return parsed, stats

# Viability filter (train+test)

def _load_step2_column_stats(run_dir: Path) -> Optional[pd.DataFrame]:
    p = run_dir / "step2_column_stats.json"
    if not p.exists():
        return None
    try:
        js = _read_json(p)
        cols = js.get("columns", {})
        if not isinstance(cols, dict) or not cols:
            return None
        rows = []
        for c, v in cols.items():
            if not isinstance(v, dict):
                continue
            rows.append(
                {
                    "col": c,
                    "train_missing": float(v.get("train_missing", np.nan)),
                    "test_missing": float(v.get("test_missing", np.nan)),
                    "train_nunique_nonmiss": int(v.get("train_nunique_nonmiss", -1)),
                    "test_nunique_nonmiss": int(v.get("test_nunique_nonmiss", -1)),
                }
            )
        df = pd.DataFrame(rows)
        return None if df.empty else df
    except Exception:
        return None


def _feature_viability_report(
    train: pd.DataFrame,
    test: pd.DataFrame,
    feature_cols: List[str],
    *,
    near_const_thresh: Optional[float],
    near_const_max_nunique: int,
    step2_stats: Optional[pd.DataFrame],
) -> pd.DataFrame:
    rep: pd.DataFrame = pd.DataFrame()

    if step2_stats is not None:
        st = step2_stats.set_index("col")
        if all(c in st.index for c in feature_cols):
            rep = st.loc[
                feature_cols,
                ["train_missing", "test_missing", "train_nunique_nonmiss", "test_nunique_nonmiss"],
            ].reset_index()

    if rep.empty:
        tr_miss = train[feature_cols].isna().mean()
        te_miss = test[feature_cols].isna().mean()
        tr_nuniq = train[feature_cols].nunique(dropna=True)
        te_nuniq = test[feature_cols].nunique(dropna=True)
        rep = pd.DataFrame(
            {
                "col": feature_cols,
                "train_missing": tr_miss.values.astype(float),
                "test_missing": te_miss.values.astype(float),
                "train_nunique_nonmiss": tr_nuniq.values.astype(int),
                "test_nunique_nonmiss": te_nuniq.values.astype(int),
            }
        )

    reasons: List[str] = []
    for _, r in rep.iterrows():
        rr: List[str] = []
        if float(r["train_missing"]) == 1.0:
            rr.append("train_all_missing")
        if float(r["test_missing"]) == 1.0:
            rr.append("test_all_missing")
        if int(r["train_nunique_nonmiss"]) <= 1:
            rr.append("train_constant_or_empty")
        if int(r["test_nunique_nonmiss"]) <= 1:
            rr.append("test_constant_or_empty")
        reasons.append("|".join(rr))
    rep["drop_reason"] = reasons

    if near_const_thresh is not None:
        thr = float(near_const_thresh)
        if not (0.0 <= thr <= 1.0):
            raise ValueError("near_const_thresh must be in [0, 1]")

        cand_mask = (
            (rep["test_missing"] < 1.0)
            & (rep["test_nunique_nonmiss"] > 1)
            & (rep["test_nunique_nonmiss"] <= int(near_const_max_nunique))
        )
        for c in rep.loc[cand_mask, "col"].tolist():
            vc = test[c].dropna().value_counts(normalize=True)
            if len(vc) and float(vc.iloc[0]) >= thr:
                i = rep.index[rep["col"] == c][0]
                prev = str(rep.loc[i, "drop_reason"]).strip("|")
                rep.loc[i, "drop_reason"] = (prev + "|" + f"test_near_constant>={thr}").strip("|")

    return rep


def _infer_cat_num(features: List[str], train: pd.DataFrame) -> Tuple[List[str], List[str]]:
    cat_cols = [
        c
        for c in features
        if pd.api.types.is_object_dtype(train[c]) or pd.api.types.is_string_dtype(train[c])
    ]
    num_cols = [c for c in features if c not in cat_cols]
    return cat_cols, num_cols


# Main Step 3

def preprocess_step3(
    *,
    train_raw: pd.DataFrame,
    test_raw: pd.DataFrame,
    core: CFGCore,
    prep: CFGPrep,
    run_dir: Path,
    features_master: Optional[List[str]] = None,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str], List[str], List[str], pd.DataFrame]:
    ID, TGT = core.id_col, core.target_col

    if ID not in train_raw.columns or ID not in test_raw.columns:
        raise ValueError(f"Missing id_col='{ID}' in train/test")
    if TGT not in train_raw.columns:
        raise ValueError(f"Missing target_col='{TGT}' in train")
    if TGT in test_raw.columns:
        raise ValueError("Target must not be present in test")

    train = train_raw.copy()
    test = test_raw.copy()

    if features_master is None:
        features_master = [c for c in train.columns if c not in (ID, TGT)]
    else:
        missing = [c for c in features_master if c not in train.columns or c not in test.columns]
        if missing:
            raise ValueError(f"features_master has missing cols (examples): {missing[:20]}")

    train = train.loc[:, [ID, TGT] + list(features_master)]
    test = test.loc[:, [ID] + list(features_master)]

    # 1) strings + nan-like cleanup
    nan_like_upper = {str(x).upper() for x in getattr(prep, "nan_like", ())}
    _normalize_strings_inplace(
        train,
        _infer_string_cols(train, exclude={TGT}),
        nan_like_upper=nan_like_upper,
        collapse_spaces=True,
    )
    _normalize_strings_inplace(
        test,
        _infer_string_cols(test, exclude=set()),
        nan_like_upper=nan_like_upper,
        collapse_spaces=True,
    )

    # 2) rating cols as normalized strings
    rating_col = str(getattr(prep, "rating_col", "рейтинг"))
    dop_col = str(getattr(prep, "doprating_col", "допрейтинг"))
    for c in (rating_col, dop_col):
        if c in train.columns and c in test.columns:
            train[c] = train[c].astype("string").str.upper()
            test[c] = test[c].astype("string").str.upper()

    # 3) doprating -> (num, letter)
    dop_features_added = False
    if dop_col in train.columns and dop_col in test.columns:
        num_name = str(getattr(prep, "doprating_num_col", "допрейтинг_num"))
        let_name = str(getattr(prep, "doprating_letter_col", "допрейтинг_letter"))
        for df in (train, test):
            df[num_name] = pd.to_numeric(
                df[dop_col].astype("string").str.extract(r"(\d+)", expand=False),
                errors="coerce",
            )
            df[let_name] = (
                df[dop_col].astype("string").str.extract(r"([A-ZА-ЯЁ])", expand=False).astype("string")
            )
        dop_features_added = True

    # 4) binary-like to Int8
    converted_binary: List[str] = []
    if bool(getattr(prep, "convert_binary_strings", True)):
        true_set = {"ДА", "TRUE", "YES", "Y", "1", "ИСТИНА"}
        false_set = {"НЕТ", "FALSE", "NO", "N", "0", "ЛОЖЬ"}
        unknown_set = set(getattr(prep, "unknown_tokens", ())) if bool(getattr(prep, "unknown_as_na", True)) else set()

        cand = [
            c
            for c in features_master
            if (c in train.columns and c in test.columns)
            and (pd.api.types.is_object_dtype(train[c]) or pd.api.types.is_string_dtype(train[c]))
        ]
        converted_binary = _convert_binary_like(
            train,
            test,
            cand,
            true_set=true_set,
            false_set=false_set,
            unknown_set=unknown_set,
            require_both_classes=True,
        )

    # 5) dates -> year/month/ym
    parsed_dates: List[str] = []
    date_stats: Dict[str, Any] = {}
    if bool(getattr(prep, "parse_dates", True)):
        explicit = list(getattr(prep, "date_cols_explicit", ()))
        prefixes = tuple(getattr(prep, "date_prefixes", ("дата_",)))

        cand = set(explicit)
        for c in features_master:
            if any(str(c).startswith(p) for p in prefixes):
                cand.add(c)

        date_cols = sorted([c for c in cand if c in train.columns and c in test.columns])
        parsed_dates, date_stats = _parse_dates_expand(
            train,
            test,
            date_cols,
            drop_raw=bool(getattr(prep, "drop_raw_dates", True)),
            min_parse_rate=float(getattr(prep, "date_min_parse_rate", 0.01)),
            formats=getattr(prep, "date_formats", None),
        )

    # 6) viability filter
    step2_stats = _load_step2_column_stats(run_dir)
    feat_cols0 = [c for c in train.columns if c not in (ID, TGT)]
    report = _feature_viability_report(
        train,
        test,
        feat_cols0,
        near_const_thresh=getattr(prep, "near_const_thresh", None),
        near_const_max_nunique=int(getattr(prep, "near_const_max_nunique", 20)),
        step2_stats=step2_stats,
    )
    drop_cols = sorted(report.loc[report["drop_reason"] != "", "col"].tolist())
    if drop_cols:
        train.drop(columns=drop_cols, inplace=True, errors="ignore")
        test.drop(columns=drop_cols, inplace=True, errors="ignore")

    # 7) final features + alignment
    features = [c for c in train.columns if c not in (ID, TGT)]
    test_feat = [c for c in test.columns if c != ID]
    if set(features) != set(test_feat):
        only_tr = sorted(list(set(features) - set(test_feat)))
        only_te = sorted(list(set(test_feat) - set(features)))
        raise ValueError(
            "Train/test feature mismatch after Step 3. "
            f"only_in_train(examples)={only_tr[:15]} | only_in_test(examples)={only_te[:15]}"
        )

    # 8) row missingness
    train["n_missing_row"] = train[features].isna().sum(axis=1).astype("int16")
    test["n_missing_row"] = test[features].isna().sum(axis=1).astype("int16")
    features = features + ["n_missing_row"]

    cat_cols, num_cols = _infer_cat_num(features, train)

    # sanity: no all-null features
    all_null_tr = train[features].isna().all()
    if bool(all_null_tr.any()):
        bad = all_null_tr[all_null_tr].index.tolist()
        raise ValueError(f"All-null feature remained in train (examples): {bad[:20]}")
    all_null_te = test[features].isna().all()
    if bool(all_null_te.any()):
        bad = all_null_te[all_null_te].index.tolist()
        raise ValueError(f"All-null feature remained in test (examples): {bad[:20]}")

    # Artifacts
    report.to_csv(run_dir / "step3_feature_report.csv", index=False, encoding="utf-8")
    _write_json(run_dir / "step3_drop_cols.json", {"drop_cols": drop_cols})
    _write_json(run_dir / "step3_features.json", {"features": features, "cat_cols": cat_cols, "num_cols": num_cols})
    _write_json(
        run_dir / "step3_meta.json",
        {
            "n_features": int(len(features)),
            "n_cat": int(len(cat_cols)),
            "n_num": int(len(num_cols)),
            "features_master_in": list(features_master),
            "features_out": list(features),
            "dropped_raw_date_cols": list(parsed_dates) if bool(getattr(prep, "drop_raw_dates", True)) else [],
            "converted_binary_cols": list(converted_binary),
            "dop_features_added": bool(dop_features_added),
            "date_parse_stats": date_stats,
            "used_step2_column_stats": bool(step2_stats is not None),
        },
    )

    if verbose:
        print("=== Step 3 OK (NO profession) ===")
        print(f"features: {len(features)} | cat: {len(cat_cols)} | num: {len(num_cols)}")
        if drop_cols:
            print("dropped:", len(drop_cols), "| examples:", drop_cols[:10], "...")
        if converted_binary:
            print("binary->Int8:", len(converted_binary), "| examples:", converted_binary[:10], "...")
        if parsed_dates:
            print("dates parsed:", len(parsed_dates), "| examples:", parsed_dates[:10], "...")

    return train, test, features, cat_cols, num_cols, report


# Run Step 3
features_master = None
p_master = RUN_DIR / "step2_features_master.json"
if p_master.exists():
    features_master = _read_json(p_master).get("features_master")

train, test, features, cat_cols, num_cols, feature_report = preprocess_step3(
    train_raw=train_raw,
    test_raw=test_raw,
    core=core,
    prep=prep,
    run_dir=RUN_DIR,
    features_master=features_master,
    verbose=run_cfg.verbose,
)


=== Step 3 OK (NO profession) ===
features: 105 | cat: 16 | num: 89
dropped: 7 | examples: ['дата_следующей_выплаты', 'код_политики', 'коэфф_невыплаченного_сумм_остатка', 'непогашенная_сумма_из_тела_займов', 'особая_ситуация', 'пени_за_дефолт', 'платежный_график'] ...
binary->Int8: 1 | examples: ['юридический_статус'] ...
dates parsed: 1 | examples: ['дата_первого_займа'] ...


In [4]:
# # 4 Profession Features (pymorphy3: lemma + canonical surface)
# # There's a lot of garbage categories in profession (~23000). Best way to clean is LLM, but I can't use it. Another option is WordNet-Ru, but WordNet has weak support for Russian language. I tried to use pymorphy3, but It's still awfull.
# # In a nutshell, split профессия_заемщика to профессия , домен профессии, статус заемщика в профессии ( profession_role, profession_domain, profession seniority).

# from __future__ import annotations

# from functools import lru_cache
# from pathlib import Path
# from typing import Any, Dict, List, Optional, Tuple
# from collections import Counter

# import re

# import pandas as pd


# # small utils
# _KEEP_AS_IS = {"__MISSING__", "__NONE__", "__GENERIC__", "__OTHER__", "__RARE__"}
# _RU_HAS_LETTER_RE = re.compile(r"[А-ЯЁа-яё]")
# _SPACE_RE = re.compile(r"\s+")

# def _normalize_profession_series(
#     s: pd.Series,
#     *,
#     missing_token: str = "__MISSING__",
#     max_len: int | None = None,
# ) -> pd.Series:
#     """Normalize raw profession into a stable categorical string."""
#     out = s.astype("string").str.strip()

#     # Real missing -> missing_token. Keep "__NONE__" intact.
#     out = out.where(out.notna() & (out != ""), other=missing_token)
#     keep = out.isin([missing_token, "__NONE__"])

#     txt = out.where(~keep, other=pd.NA)
#     txt = (txt.str.lower()
#               .str.replace("ё", "е", regex=False)
#               .str.replace(r"[^0-9a-zа-я\s\-\./]", " ", regex=True)
#               .str.replace(r"\s+", " ", regex=True)
#               .str.strip()
#           )

#     if max_len is not None and max_len > 0:
#         txt = txt.str.slice(0, int(max_len))

#     out = out.where(keep, other=txt).fillna(missing_token)
#     return out.astype("string")

# def _sanitize_cat_series(s: pd.Series, *, missing_token: str) -> pd.Series:
#     x = s.astype("string").str.strip()
#     x = x.where(x.notna() & (x != ""), other=missing_token)
#     return x.astype("string")


# def _strip_prefix_safe(s: pd.Series, *, prefix: str) -> pd.Series:
#     x = s.astype("string").str.strip()
#     if prefix:
#         if hasattr(x.str, "removeprefix"):
#             x = x.str.removeprefix(prefix)
#         else:
#             x = x.str.replace(prefix, "", regex=False)
#     return x.astype("string")


# def _try_init_pymorphy3() -> Tuple[Optional[Any], str]:
#     try:
#         import pymorphy3  # type: ignore
#         m = pymorphy3.MorphAnalyzer(lang="ru")
#         ver = getattr(pymorphy3, "__version__", "unknown")
#         return m, f"pymorphy3={ver}"
#     except Exception as e:
#         return None, f"unavailable: {type(e).__name__}: {e}"


# def _finalize_role_domain(
#     s: pd.Series,
#     *,
#     prof_missing_mask: pd.Series,
#     missing_token: str,
#     none_token: str,
# ) -> pd.Series:
#     # builder sometimes yields <NA>/empty; treat as __NONE__ if profession exists
#     x = s.astype("string").str.strip()
#     x = x.where(x.notna() & (x != ""), other="<NA>")
#     x = x.replace({"<NA>": none_token, "<na>": none_token})
#     x = x.mask(prof_missing_mask.astype(bool), other=missing_token)

#     # stabilize casing for non-service labels
#     x = x.where(x.isin(_KEEP_AS_IS), other=x.astype("string").str.upper())
#     return x.astype("string")


# def _build_lemma2canon(
#     train_s: pd.Series,
#     *,
#     morph: Any,
#     protected: set[str],
# ) -> Dict[str, str]:
#     """
#     lemma -> canonical surface (train-driven).
#     Prefer lemma itself if it appears as a surface.
#     """
#     tr = train_s.astype("string").fillna("").str.strip()
#     tr = tr.where(tr != "", other="")

#     vc = tr.value_counts(dropna=False)

#     @lru_cache(maxsize=200_000)
#     def _lemma_of(surface: str) -> str:
#         s = str(surface).strip()
#         if not s or s in protected:
#             return s
#         if not _RU_HAS_LETTER_RE.search(s):
#             return s.upper()

#         parts = [p for p in _SPACE_RE.split(s) if p]
#         out: List[str] = []
#         for p in parts:
#             if p in protected:
#                 out.append(p)
#                 continue
#             if not _RU_HAS_LETTER_RE.search(p):
#                 out.append(p.upper())
#                 continue
#             parses = morph.parse(p.lower())
#             out.append(str(parses[0].normal_form).upper() if parses else p.upper())
#         return " ".join(out)

#     # Count surfaces per lemma
#     lemma2surf: Dict[str, Dict[str, int]] = {}
#     for surf, cnt in zip(vc.index.astype("string").tolist(), vc.values.tolist()):
#         s = str(surf)
#         lem = _lemma_of(s)
#         d = lemma2surf.setdefault(lem, {})
#         d[s] = d.get(s, 0) + int(cnt)

#     lemma2canon: Dict[str, str] = {}
#     for lem, surf_cnt in lemma2surf.items():
#         if lem in surf_cnt:
#             lemma2canon[lem] = lem
#             continue
#         best = sorted(surf_cnt.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
#         lemma2canon[lem] = best

#     return lemma2canon


# def _lemmatize_and_canonicalize(
#     train_s: pd.Series,
#     test_s: pd.Series,
#     *,
#     morph: Any,
#     protected: set[str],
# ) -> Tuple[pd.Series, pd.Series, Dict[str, Any]]:
#     """
#     surface -> lemma -> canonical surface (chosen from train).
#     """
#     lemma2canon = _build_lemma2canon(train_s, morph=morph, protected=protected)

#     @lru_cache(maxsize=200_000)
#     def _lemma_of(surface: str) -> str:
#         s = str(surface).strip()
#         if not s or s in protected:
#             return s
#         if not _RU_HAS_LETTER_RE.search(s):
#             return s.upper()

#         parts = [p for p in _SPACE_RE.split(s) if p]
#         out: List[str] = []
#         for p in parts:
#             if p in protected:
#                 out.append(p)
#                 continue
#             if not _RU_HAS_LETTER_RE.search(p):
#                 out.append(p.upper())
#                 continue
#             parses = morph.parse(p.lower())
#             out.append(str(parses[0].normal_form).upper() if parses else p.upper())
#         return " ".join(out)

#     def _apply(s: pd.Series) -> pd.Series:
#         x = s.astype("string")
#         lem = x.map(lambda z: _lemma_of("" if pd.isna(z) else str(z)))
#         can = lem.map(lambda z: lemma2canon.get(str(z), str(z)))
#         return can.astype("string")

#     tr2 = _apply(train_s)
#     te2 = _apply(test_s)

#     meta = {
#         "n_lemma2canon": int(len(lemma2canon)),
#         "lemma2canon_head": dict(list(lemma2canon.items())[:30]),
#     }
#     return tr2, te2, meta

# # Seniority regexes
# _SEN_PATTERNS = [
#     ("INTERN",  r"\b(?:интерн|intern|стажер|стажёр|trainee)\b"),
#     ("JUNIOR",  r"\b(?:junior|джун|младш(?:ий|ая)|ассистент|assistant)\b"),
#     ("MIDDLE",  r"\b(?:middle|мидл|специалист)\b"),
#     ("SENIOR",  r"\b(?:senior|сеньор|старш(?:ий|ая)|ведущ(?:ий|ая)|lead|лид)\b"),
#     ("MANAGER", r"\b(?:руковод(?:итель|ительница)|начальник|директор|manager|head)\b"),
# ]

# def extract_seniority_from_norm(
#     s: pd.Series,
#     *,
#     missing_token: str = "__MISSING__",
#     none_token: str = "__NONE__",
# ) -> pd.Series:
#     """
#     Extract a coarse seniority label from normalized profession text.

#     Output categories:
#       - missing_token for real missing
#       - none_token for explicit "__NONE__"
#       - one of: INTERN/JUNIOR/MIDDLE/SENIOR/MANAGER
#       - "__GENERIC__" if nothing matched
#     """
#     x = s.astype("string").str.strip()

#     # Preserve special tokens as-is (policy compatible with Step 5)
#     is_missing = x.eq(missing_token) | x.isna() | (x == "")
#     is_none = x.eq(none_token) | x.eq("__NONE__")

#     out = pd.Series(pd.NA, index=x.index, dtype="string")

#     # Work only on real text rows
#     txt = x.mask(is_missing | is_none, other=pd.NA)

#     # Default for non-matched text
#     out = out.where(out.notna(), other="__GENERIC__")

#     # Apply patterns in priority order
#     # (first match wins)
#     for label, rx in _SEN_PATTERNS:
#         m = txt.str.contains(rx, case=False, regex=True, na=False)
#         out = out.mask(m, other=label)

#     # Put special tokens back
#     out = out.mask(is_none, other=none_token)
#     out = out.mask(is_missing, other=missing_token)

#     return out.astype("string")

# _TOKEN_RE = re.compile(r"[0-9a-zа-яё]+", re.IGNORECASE)

# def build_profession_role_domain(
#     train_norm: pd.Series,
#     test_norm: pd.Series,
#     *,
#     missing_token: str = "__MISSING__",
#     role_other: str = "__OTHER__",
#     domain_other: str = "__OTHER__",
#     generic_token: str = "__GENERIC__",
#     min_df_role: int = 100,
#     min_df_domain: int = 500,
#     min_df_domain_fallback: int = 100,
#     min_token_len: int = 4,
#     prefix_role: str = "ROLE_",
#     prefix_domain: str = "DOM_",
#     max_df_share: float = 0.30,
#     apply_stemming: bool = False,
# ) -> Tuple[pd.Series, pd.Series, pd.Series, pd.Series, Dict[str, Any]]:
#     """
#     Build two derived categoricals from normalized profession:
#       - role: more specific token (rarer but not too rare)
#       - domain: more general token (more frequent but not too generic)

#     No rare bucketing here.
#     """

#     def _tokenize_one(x: str) -> List[str]:
#         if not x or x in (missing_token, "__NONE__"):
#             return []
#         toks = _TOKEN_RE.findall(x.lower().replace("ё", "е"))
#         toks = [t for t in toks if len(t) >= int(min_token_len)]
#         if not toks:
#             return []
#         if apply_stemming:
#             # very light suffix chopping
#             def _stem_ru(t: str) -> str:
#                 for suf in ("ами","ями","ого","ему","ими","ыми","ая","яя","ое","ее","ий","ый","ой","ие","ые","ых","ам","ям","ах","ях","ом","ем","ов","ев","а","я","о","е","ы","и","у","ю"):
#                     if len(t) > len(suf) + 2 and t.endswith(suf):
#                         return t[: -len(suf)]
#                 return t
#             toks = [_stem_ru(t) for t in toks]
#         return toks

#     def _build_df_counts(s: pd.Series) -> Counter:
#         df = Counter()
#         for v in s.astype("string").fillna("").tolist():
#             x = str(v).strip()
#             if (not x) or x == missing_token or x == "__NONE__":
#                 continue
#             toks = set(_tokenize_one(x))
#             for t in toks:
#                 df[t] += 1
#         return df

#     tr = train_norm.astype("string").fillna("").str.strip()
#     te = test_norm.astype("string").fillna("").str.strip()

#     n_train = int(len(tr))
#     df_counts = _build_df_counts(tr)

#     # Filter: drop too common tokens
#     max_df = int(max(1, round(float(max_df_share) * n_train)))
#     valid = {t: c for t, c in df_counts.items() if c <= max_df}

#     # Candidates for role/domain
#     role_cand = {t: c for t, c in valid.items() if c >= int(min_df_role)}
#     dom_cand  = {t: c for t, c in valid.items() if c >= int(min_df_domain)}
#     dom_fallback = {t: c for t, c in valid.items() if c >= int(min_df_domain_fallback)}

#     def _pick_role(toks: List[str]) -> str:
#         # Prefer the rarest token among role_cand (more specific)
#         cands = [(role_cand.get(t, 0), t) for t in toks if t in role_cand]
#         if not cands:
#             return role_other
#         cands.sort(key=lambda z: (z[0], z[1]))  # rarest first
#         return prefix_role + cands[0][1]

#     def _pick_domain(toks: List[str]) -> str:
#         # Prefer the most frequent token among dom_cand (more stable)
#         cands = [(dom_cand.get(t, 0), t) for t in toks if t in dom_cand]
#         if cands:
#             cands.sort(key=lambda z: (-z[0], z[1]))  # most frequent first
#             return prefix_domain + cands[0][1]

#         # Fallback with lower threshold
#         c2 = [(dom_fallback.get(t, 0), t) for t in toks if t in dom_fallback]
#         if c2:
#             c2.sort(key=lambda z: (-z[0], z[1]))
#             return prefix_domain + c2[0][1]

#         # If any tokens exist but none passed thresholds
#         return generic_token

#     def _apply(s: pd.Series) -> Tuple[pd.Series, pd.Series]:
#         role_out = []
#         dom_out = []
#         for v in s.astype("string").fillna("").tolist():
#             x = str(v).strip()
#             if (not x) or x == missing_token:
#                 role_out.append(missing_token)
#                 dom_out.append(missing_token)
#                 continue
#             if x == "__NONE__":
#                 # profession exists but explicitly none
#                 role_out.append("__NONE__")
#                 dom_out.append("__NONE__")
#                 continue
#             toks = _tokenize_one(x)
#             if not toks:
#                 role_out.append(generic_token)
#                 dom_out.append(generic_token)
#                 continue
#             role_out.append(_pick_role(toks))
#             dom_out.append(_pick_domain(toks))
#         return pd.Series(role_out, dtype="string"), pd.Series(dom_out, dtype="string")

#     role_tr, dom_tr = _apply(tr)
#     role_te, dom_te = _apply(te)

#     meta = {
#         "n_train": n_train,
#         "min_df_role": int(min_df_role),
#         "min_df_domain": int(min_df_domain),
#         "min_df_domain_fallback": int(min_df_domain_fallback),
#         "min_token_len": int(min_token_len),
#         "max_df_share": float(max_df_share),
#         "apply_stemming": bool(apply_stemming),
#         "n_tokens_df": int(len(df_counts)),
#         "n_tokens_valid": int(len(valid)),
#         "n_role_tokens": int(len(role_cand)),
#         "n_domain_tokens": int(len(dom_cand)),
#         "n_domain_fallback_tokens": int(len(dom_fallback)),
#         "role_top10": dict(sorted(role_cand.items(), key=lambda kv: (-kv[1], kv[0]))[:10]),
#         "domain_top10": dict(sorted(dom_cand.items(), key=lambda kv: (-kv[1], kv[0]))[:10]),
#     }

#     return role_tr, role_te, dom_tr, dom_te, meta    

# def preprocess_step4_profession(
#     *,
#     train: pd.DataFrame,
#     test: pd.DataFrame,
#     core: CFGCore,
#     prep: CFGPrep,
#     run_dir: Path,
#     verbose: bool = True,
# ) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
#     ID, TGT = core.id_col, core.target_col
#     if ID not in train.columns or ID not in test.columns:
#         raise ValueError("Step4: missing id column in train/test")
#     if TGT not in train.columns:
#         raise ValueError("Step4: missing target column in train")

#     prof_col = str(getattr(prep, "prof_col", "профессия_заемщика"))

#     if not bool(getattr(prep, "use_profession_step4", True)):
#         if verbose:
#             print("=== Step 4 (profession) skipped: use_profession_step4=False ===")
#         return train, test, {"enabled": False, "reason": "disabled_by_config"}

#     if prof_col not in train.columns or prof_col not in test.columns:
#         if verbose:
#             print(f"=== Step 4 (profession) skipped: '{prof_col}' not found ===")
#         return train, test, {"enabled": False, "reason": "prof_col_missing", "prof_col": prof_col}

#     morph, backend = _try_init_pymorphy3()
#     if morph is None:
#         raise RuntimeError(
#             "Step4: pymorphy3 init failed. "
#             f"Status: {backend}. Install: pip install pymorphy3 pymorphy3-dicts-ru"
#         )

#     missing_token = str(getattr(prep, "missing_token", "__MISSING__"))
#     rare_token = str(getattr(prep, "rare_token", "__RARE__"))
#     none_token = str(getattr(prep, "prof_none_token", "__NONE__"))

#     norm_col = f"{prof_col}_норм"
#     superclass_col = str(getattr(prep, "prof_superclass_col", "профессия_superclass"))
#     seniority_col  = str(getattr(prep, "prof_seniority_col",  "профессия_seniority"))
#     domain_col     = str(getattr(prep, "prof_domain_col",     "профессия_domain"))

#     # 1) normalize raw profession
#     norm_max_len = getattr(prep, "prof_normalize_max_len", None)
#     norm_max_len = None if norm_max_len in (None, 0, -1, False) else int(norm_max_len)

#     train[norm_col] = _normalize_profession_series(train[prof_col], missing_token=missing_token, max_len=norm_max_len)
#     test[norm_col]  = _normalize_profession_series(test[prof_col],  missing_token=missing_token, max_len=norm_max_len)

#     train[norm_col] = _sanitize_cat_series(train[norm_col], missing_token=missing_token)
#     test[norm_col]  = _sanitize_cat_series(test[norm_col],  missing_token=missing_token)

#     tr_prof_missing = train[norm_col].astype("string").eq(missing_token)
#     te_prof_missing = test[norm_col].astype("string").eq(missing_token)

#     # 2) seniority
#     train[seniority_col] = _sanitize_cat_series(
#         extract_seniority_from_norm(train[norm_col], missing_token=missing_token),
#         missing_token=missing_token,
#     )
#     test[seniority_col] = _sanitize_cat_series(
#         extract_seniority_from_norm(test[norm_col], missing_token=missing_token),
#         missing_token=missing_token,
#     )

#     # 3) role/domain
#     rd_params = {
#         "min_df_role": int(getattr(prep, "prof_role_min_df", 100)),
#         "min_df_domain": int(getattr(prep, "prof_domain_min_df", 500)),
#         "min_df_domain_fallback": int(getattr(prep, "prof_domain_min_df_fallback", 100)),
#         "min_token_len": int(getattr(prep, "prof_sc_min_token_len", 4)),
#         "max_df_share": float(getattr(prep, "prof_sc_max_df_share", 0.30)),
#         "apply_stemming": bool(getattr(prep, "prof_super_apply_stemming", True)),
#         "role_other": str(getattr(prep, "prof_role_other_token", "__OTHER__")),
#         "domain_other": str(getattr(prep, "prof_domain_other_token", "__OTHER__")),
#         "generic_token": str(getattr(prep, "prof_super_generic_token", "__GENERIC__")),
#         "prefix_role": str(getattr(prep, "prof_role_prefix", "ROLE_")),
#         "prefix_domain": str(getattr(prep, "prof_domain_prefix", "DOM_")),
#     }

#     role_tr, role_te, dom_tr, dom_te, rd_meta = build_profession_role_domain(
#         train[norm_col],
#         test[norm_col],
#         missing_token=missing_token,
#         role_other=rd_params["role_other"],
#         domain_other=rd_params["domain_other"],
#         generic_token=rd_params["generic_token"],
#         min_df_role=rd_params["min_df_role"],
#         min_df_domain=rd_params["min_df_domain"],
#         min_df_domain_fallback=rd_params["min_df_domain_fallback"],
#         min_token_len=rd_params["min_token_len"],
#         prefix_role=rd_params["prefix_role"],
#         prefix_domain=rd_params["prefix_domain"],
#         max_df_share=rd_params["max_df_share"],
#         apply_stemming=rd_params["apply_stemming"],
#     )

#     role_tr = _strip_prefix_safe(role_tr, prefix=rd_params["prefix_role"])
#     role_te = _strip_prefix_safe(role_te, prefix=rd_params["prefix_role"])
#     dom_tr  = _strip_prefix_safe(dom_tr,  prefix=rd_params["prefix_domain"])
#     dom_te  = _strip_prefix_safe(dom_te,  prefix=rd_params["prefix_domain"])

#     role_tr = _finalize_role_domain(role_tr, prof_missing_mask=tr_prof_missing, missing_token=missing_token, none_token=none_token)
#     role_te = _finalize_role_domain(role_te, prof_missing_mask=te_prof_missing, missing_token=missing_token, none_token=none_token)
#     dom_tr  = _finalize_role_domain(dom_tr,  prof_missing_mask=tr_prof_missing, missing_token=missing_token, none_token=none_token)
#     dom_te  = _finalize_role_domain(dom_te,  prof_missing_mask=te_prof_missing, missing_token=missing_token, none_token=none_token)

#     # 3b) lemma + canon
#     protected = set(_KEEP_AS_IS)
#     role_tr2, role_te2, role_lemma_meta = _lemmatize_and_canonicalize(role_tr, role_te, morph=morph, protected=protected)
#     dom_tr2,  dom_te2,  dom_lemma_meta  = _lemmatize_and_canonicalize(dom_tr,  dom_te,  morph=morph, protected=protected)

#     # safety: enforce missing policy after canon
#     role_tr2 = role_tr2.mask(tr_prof_missing, other=missing_token)
#     role_te2 = role_te2.mask(te_prof_missing, other=missing_token)
#     dom_tr2  = dom_tr2.mask(tr_prof_missing, other=missing_token)
#     dom_te2  = dom_te2.mask(te_prof_missing, other=missing_token)

#     train[superclass_col] = _sanitize_cat_series(role_tr2, missing_token=missing_token)
#     test[superclass_col]  = _sanitize_cat_series(role_te2, missing_token=missing_token)
#     train[domain_col]     = _sanitize_cat_series(dom_tr2,  missing_token=missing_token)
#     test[domain_col]      = _sanitize_cat_series(dom_te2,  missing_token=missing_token)

#     # 4) optional bucketing
#     rare_stats: Dict[str, Any] = {}

#     if bool(getattr(prep, "prof_bucket_superclass", False)):
#         tr_b, te_b, st = _rare_bucket_series(
#             train[superclass_col], test[superclass_col],
#             rare_token=rare_token,
#             target_keep_share=float(getattr(prep, "prof_superclass_keep_share", 0.995)),
#             min_count_floor=int(getattr(prep, "prof_superclass_min_floor", 10)),
#             max_count_cap=int(getattr(prep, "prof_superclass_max_cap", 500)),
#             fixed_min_count=None,
#         )
#         train[superclass_col] = _sanitize_cat_series(tr_b, missing_token=missing_token)
#         test[superclass_col]  = _sanitize_cat_series(te_b, missing_token=missing_token)
#         rare_stats[superclass_col] = st

#     if bool(getattr(prep, "prof_bucket_domain", False)):
#         tr_b, te_b, st = _rare_bucket_series(
#             train[domain_col], test[domain_col],
#             rare_token=rare_token,
#             target_keep_share=float(getattr(prep, "prof_domain_keep_share", 0.995)),
#             min_count_floor=int(getattr(prep, "prof_domain_min_floor", 10)),
#             max_count_cap=int(getattr(prep, "prof_domain_max_cap", 1000)),
#             fixed_min_count=None,
#         )
#         train[domain_col] = _sanitize_cat_series(tr_b, missing_token=missing_token)
#         test[domain_col]  = _sanitize_cat_series(te_b, missing_token=missing_token)
#         rare_stats[domain_col] = st

#     # 5) drop raw/norm
#     drop_raw = bool(getattr(prep, "drop_raw_profession", True))
#     drop_norm = bool(getattr(prep, "drop_norm_profession", True))

#     drop_cols: List[str] = []
#     if drop_raw and prof_col in train.columns:
#         drop_cols.append(prof_col)
#     if drop_norm and norm_col in train.columns:
#         drop_cols.append(norm_col)

#     if drop_cols:
#         train.drop(columns=drop_cols, inplace=True, errors="ignore")
#         test.drop(columns=drop_cols, inplace=True, errors="ignore")

#     # 6) n_missing_row
#     if "n_missing_row" in train.columns:
#         train.drop(columns=["n_missing_row"], inplace=True, errors="ignore")
#         test.drop(columns=["n_missing_row"], inplace=True, errors="ignore")

#     feat_cols = [c for c in train.columns if c not in (ID, TGT)]
#     test_feat = [c for c in test.columns if c != ID]
#     if set(feat_cols) != set(test_feat):
#         only_tr = sorted(list(set(feat_cols) - set(test_feat)))
#         only_te = sorted(list(set(test_feat) - set(feat_cols)))
#         raise ValueError(
#             "Step4: train/test feature mismatch after profession step. "
#             f"only_in_train(examples)={only_tr[:15]} | only_in_test(examples)={only_te[:15]}"
#         )

#     train["n_missing_row"] = train[feat_cols].isna().sum(axis=1).astype("int16")
#     test["n_missing_row"]  = test[feat_cols].isna().sum(axis=1).astype("int16")

#     # 7) artifacts
#     meta = {
#         "enabled": True,
#         "prof_col_raw": prof_col,
#         "norm_col_internal": norm_col,
#         "superclass_col": superclass_col,
#         "seniority_col": seniority_col,
#         "domain_col": domain_col,
#         "dropped_cols": drop_cols,
#         "normalize_max_len": norm_max_len,
#         "missing_token": missing_token,
#         "none_token": none_token,
#         "pymorph_backend": backend,
#         "role_domain_params": rd_params,
#         "role_domain_stats": rd_meta,
#         "lemma_canon": {"role": role_lemma_meta, "domain": dom_lemma_meta},
#         "rare_bucket_stats": rare_stats,
#         "final_prefixes_removed": True,
#     }

#     _write_json(run_dir / "step4_prof_role_domain.json", rd_meta)
#     _write_json(run_dir / "step4_prof_meta.json", meta)

#     if verbose:
#         print("=== Step 4 OK (profession) ===")
#         print("added:", [superclass_col, seniority_col, domain_col])
#         if drop_cols:
#             print("dropped:", drop_cols)
#         print("pymorph:", backend)
#         print("superclass_nunique_train =", int(train[superclass_col].nunique(dropna=False)))
#         print("domain_nunique_train     =", int(train[domain_col].nunique(dropna=False)))
#         print("seniority_nunique_train  =", int(train[seniority_col].nunique(dropna=False)))

#         # no '<NA>' should remain
#         for col in (superclass_col, domain_col):
#             if bool(train[col].astype("string").str.contains(r"<NA>", regex=True, na=False).any()):
#                 raise ValueError(f"Step4: found '<NA>' artifacts in {col}")

#     return train, test, meta


# # Run Step 4
# train, test, step4_prof_meta = preprocess_step4_profession(
#     train=train,
#     test=test,
#     core=core,
#     prep=prep,
#     run_dir=RUN_DIR,
#     verbose=run_cfg.verbose,
# )

# ID, TGT = core.id_col, core.target_col
# features = [c for c in train.columns if c not in (ID, TGT)]
# cat_cols, num_cols = _infer_cat_num(features, train)
# _write_json(RUN_DIR / "step4_features.json", {"features": features, "cat_cols": cat_cols, "num_cols": num_cols})

# if run_cfg.verbose:
#     print("=== Features after Step 4 ===")
#     print(f"features: {len(features)} | cat: {len(cat_cols)} | num: {len(num_cols)}")


In [5]:
# # --- Debug / QA for profession fields AFTER Step 4 ---

# import pandas as pd
# import numpy as np

# def _counts(s: pd.Series) -> pd.Series:
#     return s.astype("string").value_counts(dropna=False)

# def _show_dist(df: pd.DataFrame, col: str, topn: int = 30) -> pd.DataFrame:
#     vc = _counts(df[col])
#     pct = (vc / len(df) * 100.0)
#     out = (
#         pd.DataFrame({"count": vc, "pct": pct})
#         .reset_index()
#         .rename(columns={"index": col})
#         .sort_values(["pct", "count", col], ascending=[False, False, True])
#         .head(topn)
#     )
#     out["pct"] = out["pct"].round(4)
#     return out

# def _tail_dist(df: pd.DataFrame, col: str, bottomn: int = 30) -> pd.DataFrame:
#     vc = _counts(df[col])
#     out = (
#         vc.rename("count")
#         .reset_index()
#         .rename(columns={"index": col})
#         .sort_values(["count", col], ascending=[True, True])
#         .head(bottomn)
#     )
#     out["pct"] = (out["count"] / len(df) * 100.0).round(4)
#     return out

# def _cross_tab_top(df: pd.DataFrame, a: str, b: str, topn: int = 20) -> pd.DataFrame:
#     ct = (
#         df[[a, b]]
#         .astype("string")
#         .value_counts(dropna=False)
#         .rename("count")
#         .reset_index()
#     )
#     ct["pct"] = (ct["count"] / len(df) * 100.0).round(4)
#     return ct.sort_values(["count", a, b], ascending=[False, True, True]).head(topn)

# def _examples_for_label(
#     df: pd.DataFrame,
#     label_col: str,
#     label: str,
#     raw_col: str | None,
#     norm_col: str | None,
#     n: int = 20,
# ) -> pd.DataFrame:
#     m = df[label_col].astype("string").eq(label)
#     if not m.any():
#         cols = [c for c in [raw_col, norm_col, label_col] if c]
#         return pd.DataFrame(columns=cols)

#     cols = [c for c in [raw_col, norm_col] if c and c in df.columns] + [label_col]
#     if not cols:
#         cols = [label_col]
#     ex = df.loc[m, cols].drop_duplicates().head(n).reset_index(drop=True)
#     return ex

# def _share(df: pd.DataFrame, col: str, token: str) -> float:
#     return float(df[col].astype("string").eq(token).mean()) * 100.0

# def debug_profession_fields_step4(
#     train: pd.DataFrame,
#     test: pd.DataFrame,
#     *,
#     # Step4 column names (defaults match the Step 4 you ran)
#     superclass_col: str = "профессия_superclass",
#     domain_col: str = "профессия_domain",
#     seniority_col: str = "профессия_seniority",
#     # Optional: raw/norm (if kept for debug)
#     raw_col: str = "профессия_заемщика",
#     norm_col: str = "профессия_заемщика_норм",
#     # tokens
#     missing_token: str = "__MISSING__",
#     generic_token: str = "__GENERIC__",
#     none_token: str = "__NONE__",
#     other_token: str = "__OTHER__",
#     topn: int = 30,
#     tailn: int = 30,
#     sample_n: int = 10,
#     seed: int = 42,
# ) -> None:
#     # ---- sanity: required columns exist
#     must = [superclass_col, domain_col, seniority_col]
#     for df_name, df in [("train", train), ("test", test)]:
#         missing = [c for c in must if c not in df.columns]
#         if missing:
#             raise KeyError(f"[{df_name}] missing required profession columns: {missing}")

#     # optional columns
#     raw_present = (raw_col in train.columns) and (raw_col in test.columns)
#     norm_present = (norm_col in train.columns) and (norm_col in test.columns)

#     raw_col_use = raw_col if raw_present else None
#     norm_col_use = norm_col if norm_present else None

#     def _header(txt: str) -> None:
#         print("\n" + "=" * 90)
#         print(txt)
#         print("=" * 90)

#     # ---- 1) overall distributions + key shares
#     for df_name, df in [("TRAIN", train), ("TEST", test)]:
#         _header(f"{df_name}: SUPERCLASS distribution (top {topn})")
#         display(_show_dist(df, superclass_col, topn=topn))

#         _header(f"{df_name}: DOMAIN distribution (top {topn})")
#         display(_show_dist(df, domain_col, topn=topn))

#         _header(f"{df_name}: SENIORITY distribution (top {topn})")
#         display(_show_dist(df, seniority_col, topn=topn))

#         print(f"\n[{df_name}] nunique(superclass) = {int(df[superclass_col].nunique(dropna=False))}")
#         print(f"[{df_name}] nunique(domain)      = {int(df[domain_col].nunique(dropna=False))}")
#         print(f"[{df_name}] nunique(seniority)   = {int(df[seniority_col].nunique(dropna=False))}")

#         # shares for QA
#         for tok in [missing_token, generic_token, none_token, other_token]:
#             if tok:
#                 print(f"[{df_name}] share({superclass_col} == {tok}) = {_share(df, superclass_col, tok):.2f}%")
#                 print(f"[{df_name}] share({domain_col} == {tok})      = {_share(df, domain_col, tok):.2f}%")
#                 print(f"[{df_name}] share({seniority_col} == {tok})   = {_share(df, seniority_col, tok):.2f}%")

#     # ---- 2) tail inspection (rare labels)
#     _header(f"TRAIN: tail labels for SUPERCLASS (bottom {tailn} by frequency)")
#     display(_tail_dist(train, superclass_col, bottomn=tailn))

#     _header(f"TRAIN: tail labels for DOMAIN (bottom {tailn} by frequency)")
#     display(_tail_dist(train, domain_col, bottomn=tailn))

#     # ---- 3) cross-tabs (pair frequencies)
#     _header("TRAIN: top (SUPERCLASS, DOMAIN) pairs")
#     display(_cross_tab_top(train, superclass_col, domain_col, topn=topn))

#     _header("TRAIN: top (SUPERCLASS, SENIORITY) pairs")
#     display(_cross_tab_top(train, superclass_col, seniority_col, topn=topn))

#     _header("TRAIN: top (DOMAIN, SENIORITY) pairs")
#     display(_cross_tab_top(train, domain_col, seniority_col, topn=topn))

#     # ---- 4) examples for manual inspection (if raw/norm present)
#     rng = np.random.default_rng(seed)

#     def _sample_labels(df: pd.DataFrame, col: str, k: int) -> list[str]:
#         labels = df[col].astype("string").value_counts().head(200).index.tolist()
#         if not labels:
#             return []
#         k = min(k, len(labels))
#         idx = rng.choice(len(labels), size=k, replace=False)
#         return [labels[i] for i in idx]

#     if raw_col_use or norm_col_use:
#         _header(f"TRAIN: examples for random SUPERCLASS labels (k={sample_n})")
#         for lab in _sample_labels(train, superclass_col, sample_n):
#             print(f"\n--- {superclass_col} = {lab} ---")
#             display(_examples_for_label(train, superclass_col, lab, raw_col_use, norm_col_use, n=15))

#         _header(f"TRAIN: examples for random DOMAIN labels (k={sample_n})")
#         for lab in _sample_labels(train, domain_col, sample_n):
#             print(f"\n--- {domain_col} = {lab} ---")
#             display(_examples_for_label(train, domain_col, lab, raw_col_use, norm_col_use, n=15))

#         _header("TRAIN: examples for SENIORITY labels")
#         for lab in ["C_LEVEL", "LEAD", "SENIOR", "MIDDLE", "JUNIOR", "ASSISTANT", none_token, missing_token]:
#             if (train[seniority_col].astype("string") == lab).any():
#                 print(f"\n--- {seniority_col} = {lab} ---")
#                 display(_examples_for_label(train, seniority_col, lab, raw_col_use, norm_col_use, n=20))
#     else:
#         _header("NOTE: raw/norm profession columns were dropped; example tables are skipped.")
#         print("If you want examples (raw->labels), temporarily set drop_raw_profession=False and/or drop_norm_profession=False in CFGPrep.")

#     _header("DONE")


# # --- Call it (prefer taking names/tokens from config if available) ---
# debug_profession_fields_step4(
#     train,
#     test,
#     superclass_col=getattr(prep, "prof_superclass_col", "профессия_superclass"),
#     domain_col=getattr(prep, "prof_domain_col", "профессия_domain"),
#     seniority_col=getattr(prep, "prof_seniority_col", "профессия_seniority"),
#     raw_col=getattr(prep, "prof_col", "профессия_заемщика"),
#     norm_col=f"{getattr(prep, 'prof_col', 'профессия_заемщика')}_норм",
#     missing_token=str(getattr(prep, "missing_token", "__MISSING__")),
#     generic_token=str(getattr(prep, "prof_super_generic_token", "__GENERIC__")),
#     none_token="__NONE__",
#     other_token=str(getattr(prep, "prof_role_other_token", "__OTHER__")),  # for QA only
#     topn=30,
#     tailn=30,
#     sample_n=10,
#     seed=42,
# )


In [6]:
# # 5 Training diagnostics (feature sets, 3x fixed runs, CatBoost). 
# # It's an old version with profession_role, profession_domain, profession seniority. It works, but I decided to comment it. See comment in the beginning of the step 4.
# from __future__ import annotations

# # Knobs (edit here)

# from dataclasses import dataclass

# @dataclass(frozen=True)
# class CatBoostDiagParams:
#     # quick sanity training, not final
#     iterations: int = 400
#     learning_rate: float = 0.07
#     depth: int = 7
#     l2_leaf_reg: float = 6.0
#     random_strength: float = 1.0
#     bagging_temperature: float = 0.5
#     od_wait: int = 80

# CB_DIAG = CatBoostDiagParams()

# @dataclass(frozen=True)
# class HoldoutCfg:
#     # fixed 3 runs per set, same rows+split for all sets inside each run
#     runs: int = 3
#     run_seeds: tuple[int, int, int] = (10607, 20607, 30607)  # <- fixed
#     sample_n: int = 200_000  # 0 -> full train
#     test_size: float = 0.20
#     seed: int = 42           # kept for cache fingerprint/back-compat, not used for runs
#     verbose: int = 100

# HOLDOUT = HoldoutCfg()

# @dataclass(frozen=True)
# class CatboostSanitizeCfg:
#     # real NA/empty -> missing token; keep "__NONE__" intact
#     missing_cat_token: str = "__MISSING__"
#     num_dtype: str = "float32"

# SANITIZE = CatboostSanitizeCfg()

# HIGH_MISSING_THR: float = 0.95
# USE_CACHE: bool = True

# # Implementation

# import json
# import time
# import hashlib
# import datetime as dt
# from pathlib import Path
# from typing import Any, Dict, List, Optional, Sequence, Tuple

# import numpy as np
# import pandas as pd
# from catboost import CatBoostClassifier, Pool
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import roc_auc_score

# # Guard: profession features only (no raw profession)

# def _assert_profession_features_only(
#     *,
#     features: Sequence[str],
#     prep: Any,
# ) -> None:
#     raw_prof = str(getattr(prep, "prof_col", "профессия_заемщика"))
#     if raw_prof in set(features):
#         raise ValueError(f"Step5: raw profession '{raw_prof}' is in features (should be dropped in Step 4)")

#     must_have = [
#         str(getattr(prep, "prof_superclass_col", "профессия_superclass")),
#         str(getattr(prep, "prof_seniority_col",  "профессия_seniority")),
#         str(getattr(prep, "prof_domain_col",     "профессия_domain")),
#     ]
#     missing = [c for c in must_have if c not in set(features)]
#     if missing:
#         raise ValueError(f"Step5: missing derived profession features in 'features': {missing}")

#     print("[Step5] profession OK:", ", ".join(must_have))


# # IO

# def _write_json(path: Path, obj: Any) -> None:
#     path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


# def _read_json(path: Path) -> Dict[str, Any]:
#     return json.loads(path.read_text(encoding="utf-8"))


# # Cache

# def _stable_hash_str(s: str, n_bytes: int = 12) -> str:
#     return hashlib.blake2b(s.encode("utf-8"), digest_size=n_bytes).hexdigest()


# def _file_fingerprint(path: Path) -> str:
#     st = path.stat()
#     return f"{path.name}|size={st.st_size}|mtime_ns={st.st_mtime_ns}"


# def _cache_paths(cache_dir: Path, base: str) -> Tuple[Path, Path]:
#     return cache_dir / f"{base}.parquet", cache_dir / f"{base}.meta.json"


# def _load_cached_matrix(cache_dir: Path, base: str) -> Tuple[pd.DataFrame, np.ndarray, Dict[str, Any]]:
#     p_parq, p_meta = _cache_paths(cache_dir, base)
#     df = pd.read_parquet(p_parq)
#     meta = _read_json(p_meta)

#     if "_target" not in df.columns:
#         raise ValueError("Cache parquet is missing '_target' column")

#     y = df["_target"].astype(np.int8).to_numpy()
#     X = df.drop(columns=["_target"])
#     return X, y, meta


# def _save_cached_matrix(
#     *,
#     X_s: pd.DataFrame,
#     y: np.ndarray,
#     cache_dir: Path,
#     base: str,
#     meta: Dict[str, Any],
# ) -> None:
#     p_parq, p_meta = _cache_paths(cache_dir, base)
#     out = X_s.copy()
#     out["_target"] = y.astype(np.int8)
#     out.to_parquet(p_parq, index=False)
#     _write_json(p_meta, meta)


# # CatBoost sanitize (single source)

# def sanitize_catboost_df(
#     X: pd.DataFrame,
#     cat_cols: Sequence[str],
#     cfg: CatboostSanitizeCfg,
# ) -> pd.DataFrame:
#     X2 = X.copy()
#     cat_set = set(cat_cols)

#     for c in X2.columns:
#         if c in cat_set:
#             s = X2[c].astype("string").str.strip()
#             s = s.where(s.notna() & (s != ""), other=cfg.missing_cat_token)
#             X2[c] = s.astype(str)
#         else:
#             X2[c] = pd.to_numeric(X2[c], errors="coerce").astype(cfg.num_dtype)

#     return X2

# # Feature sets

# def _infer_cat_num(features: Sequence[str], train: pd.DataFrame) -> Tuple[List[str], List[str]]:
#     feats = list(features)
#     cat_cols = [c for c in feats if (pd.api.types.is_object_dtype(train[c]) or pd.api.types.is_string_dtype(train[c]))]
#     num_cols = [c for c in feats if c not in cat_cols]
#     return cat_cols, num_cols


# def _ensure_missing_table(
#     *,
#     features_all: Sequence[str],
#     train: pd.DataFrame,
#     test: pd.DataFrame,
#     feature_report: Optional[pd.DataFrame],
# ) -> pd.DataFrame:
#     feats = list(features_all)
#     for c in feats:
#         if c not in train.columns or c not in test.columns:
#             raise ValueError(f"Step5: feature missing in train/test: {c}")

#     rows: Dict[str, Dict[str, float]] = {}

#     if feature_report is not None:
#         need = {"col", "train_missing", "test_missing"}
#         if need.issubset(feature_report.columns):
#             rep = feature_report[["col", "train_missing", "test_missing"]].copy()
#             rep = rep.dropna(subset=["col"])
#             for _, r in rep.iterrows():
#                 col = str(r["col"])
#                 if col:
#                     rows[col] = {
#                         "train_missing": float(r["train_missing"]),
#                         "test_missing": float(r["test_missing"]),
#                     }

#     missing_cols = [c for c in feats if c not in rows]
#     if missing_cols:
#         tr_miss = train[missing_cols].isna().mean().astype(float)
#         te_miss = test[missing_cols].isna().mean().astype(float)
#         for c in missing_cols:
#             rows[c] = {"train_missing": float(tr_miss[c]), "test_missing": float(te_miss[c])}

#     miss = pd.DataFrame(
#         [{"col": c, "train_missing": v["train_missing"], "test_missing": v["test_missing"]} for c, v in rows.items()]
#     ).set_index("col")

#     uncovered = sorted(set(feats) - set(miss.index))
#     if uncovered:
#         raise ValueError(f"Step5: missing table incomplete (examples): {uncovered[:20]}")

#     return miss.loc[feats, ["train_missing", "test_missing"]]


# def build_feature_sets(
#     *,
#     features_all: Sequence[str],
#     cat_cols_all: Sequence[str],
#     num_cols_all: Sequence[str],
#     train: pd.DataFrame,
#     test: pd.DataFrame,
#     feature_report: Optional[pd.DataFrame],
#     high_missing_thr: float,
#     prep: Any,
# ) -> Dict[str, List[str]]:
#     feats_all = list(features_all)
#     feat_set = set(feats_all)

#     if not (0.0 <= float(high_missing_thr) <= 1.0):
#         raise ValueError("high_missing_thr must be in [0, 1]")

#     miss = _ensure_missing_table(
#         features_all=feats_all,
#         train=train,
#         test=test,
#         feature_report=feature_report,
#     )

#     drop_high = miss.index[
#         (miss["train_missing"] >= high_missing_thr) | (miss["test_missing"] >= high_missing_thr)
#     ].tolist()
#     drop_high_set = set(drop_high)

#     prof_domain_col = str(getattr(prep, "prof_domain_col", "профессия_domain"))
#     has_prof_domain = prof_domain_col in feat_set

#     if not has_prof_domain:
#         print(f"[Step5] note: prof_domain_col '{prof_domain_col}' not in features, skip *_wo_prof_domain sets")

#     # Build sets first (unordered).
#     sets: Dict[str, List[str]] = {}

#     sets["ALL"] = feats_all

#     if has_prof_domain:
#         sets["ALL_wo_prof_domain"] = [c for c in feats_all if c != prof_domain_col]

#     hm_name = f"ALL_wo_high_missing>={high_missing_thr:g}"
#     sets[hm_name] = [c for c in feats_all if c not in drop_high_set]

#     if has_prof_domain:
#         # High-missing set, but without profession domain.
#         sets["ALL_wo_high_missing_wo_prof_domain"] = [c for c in sets[hm_name] if c != prof_domain_col]

#     sets["NUM_ONLY"] = [c for c in num_cols_all if c in feat_set]
#     sets["CAT_ONLY"] = [c for c in cat_cols_all if c in feat_set]

#     # Basic checks.
#     for name, cols in sets.items():
#         if not cols:
#             raise ValueError(f"Feature set '{name}' is empty")
#         if len(set(cols)) != len(cols):
#             raise ValueError(f"Feature set '{name}' has duplicates")
#         bad = sorted(set(cols) - feat_set)
#         if bad:
#             raise ValueError(f"Feature set '{name}' has unknown cols (examples): {bad[:10]}")

#     # Enforce requested order.
#     order = [
#         "ALL",
#         "ALL_wo_prof_domain",
#         hm_name,
#         "ALL_wo_high_missing_wo_prof_domain",
#         "NUM_ONLY",
#         "CAT_ONLY",
#     ]
#     out: Dict[str, List[str]] = {}
#     for k in order:
#         if k in sets:
#             out[k] = sets[k]

#     # Keep any leftovers at the end (should be none, but safe).
#     for k in sets:
#         if k not in out:
#             out[k] = sets[k]

#     return out
    
# # Holdout runs
# def _cb_params(
#     *,
#     seed: int,
#     threads: int,
#     verbose: int,
#     p: CatBoostDiagParams,
# ) -> Dict[str, Any]:
#     return dict(
#         loss_function="Logloss",
#         eval_metric="AUC",
#         iterations=int(p.iterations),
#         learning_rate=float(p.learning_rate),
#         depth=int(p.depth),
#         l2_leaf_reg=float(p.l2_leaf_reg),
#         random_strength=float(p.random_strength),
#         bagging_temperature=float(p.bagging_temperature),
#         od_type="Iter",
#         od_wait=int(p.od_wait),
#         random_seed=int(seed),
#         thread_count=int(threads) if threads is not None else -1,
#         allow_writing_files=False,
#         verbose=int(verbose),
#     )


# def _random_stratified_subset_idx(y: np.ndarray, n: int, seed: int) -> np.ndarray:
#     rng = np.random.default_rng(seed)
#     y = np.asarray(y, dtype=np.int8)

#     idx0 = np.flatnonzero(y == 0)
#     idx1 = np.flatnonzero(y == 1)

#     n = int(min(n, len(y)))
#     if n <= 0:
#         raise ValueError("sample_n must be positive")

#     p1 = float(y.mean())
#     n1 = int(round(n * p1))
#     n0 = n - n1

#     n0 = min(n0, len(idx0))
#     n1 = min(n1, len(idx1))

#     take0 = rng.choice(idx0, size=n0, replace=False) if n0 else np.array([], dtype=int)
#     take1 = rng.choice(idx1, size=n1, replace=False) if n1 else np.array([], dtype=int)

#     idx = np.concatenate([take0, take1])
#     rng.shuffle(idx)
#     return idx


# def _make_run_plan(
#     *,
#     y: np.ndarray,
#     holdout_cfg: HoldoutCfg,
# ) -> List[Dict[str, Any]]:
#     # Same rows + same split for every feature set within a run.
#     y = np.asarray(y, dtype=np.int8)

#     if int(holdout_cfg.runs) != len(holdout_cfg.run_seeds):
#         raise ValueError("HoldoutCfg: runs must match len(run_seeds)")

#     plans: List[Dict[str, Any]] = []
#     for run_i, seed_run in enumerate(holdout_cfg.run_seeds, 1):
#         seed_run = int(seed_run)

#         if holdout_cfg.sample_n and int(holdout_cfg.sample_n) > 0 and int(holdout_cfg.sample_n) < len(y):
#             idx = _random_stratified_subset_idx(y, n=int(holdout_cfg.sample_n), seed=seed_run)
#         else:
#             idx = np.arange(len(y), dtype=int)

#         yr = y[idx]
#         tr_idx, va_idx = train_test_split(
#             np.arange(len(yr), dtype=int),
#             test_size=float(holdout_cfg.test_size),
#             random_state=seed_run,
#             stratify=yr,
#         )
#         plans.append({"run": run_i, "seed": seed_run, "idx": idx, "tr_idx": tr_idx, "va_idx": va_idx})

#     return plans


# def run_3x_holdout_suite(
#     *,
#     X_all_s: pd.DataFrame,
#     y: np.ndarray,
#     feature_sets: Dict[str, List[str]],
#     cat_cols_all: List[str],
#     threads: int,
#     holdout_cfg: HoldoutCfg,
#     cb_params: CatBoostDiagParams,
# ) -> pd.DataFrame:
#     rows: List[Dict[str, Any]] = []
#     y = np.asarray(y, dtype=np.int8)

#     run_plan = _make_run_plan(y=y, holdout_cfg=holdout_cfg)
#     print("[Step5] run seeds:", [p["seed"] for p in run_plan])

#     for set_name, feat_cols in feature_sets.items():
#         print("\n" + "=" * 90)
#         print(f"Feature set: {set_name} | n_feat={len(feat_cols)}")
#         print("=" * 90)

#         X_set = X_all_s[feat_cols]
#         cat_in_use = [
#             c for c in cat_cols_all
#             if c in feat_cols and c in X_set.columns and not pd.api.types.is_numeric_dtype(X_set[c])
#         ]

#         for p in run_plan:
#             idx = p["idx"]
#             tr_idx = p["tr_idx"]
#             va_idx = p["va_idx"]
#             seed_run = int(p["seed"])
#             run_i = int(p["run"])

#             Xr = X_set.iloc[idx].reset_index(drop=True)
#             yr = y[idx]

#             X_tr, X_va = Xr.iloc[tr_idx], Xr.iloc[va_idx]
#             y_tr, y_va = yr[tr_idx], yr[va_idx]

#             t0 = time.time()
#             train_pool = Pool(X_tr, y_tr, cat_features=cat_in_use)
#             valid_pool = Pool(X_va, y_va, cat_features=cat_in_use)

#             model = CatBoostClassifier(**_cb_params(
#                 seed=seed_run,
#                 threads=threads,
#                 verbose=holdout_cfg.verbose,
#                 p=cb_params,
#             ))
#             model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

#             proba = model.predict_proba(X_va)[:, 1]
#             auc = float(roc_auc_score(y_va, proba))
#             wall = time.time() - t0

#             print(f"[{set_name}][run {run_i}/{holdout_cfg.runs}] ROC_AUC={auc:.6f} | "
#                   f"rows={len(yr)} | trees={int(model.tree_count_)} | sec={wall:.1f}")

#             rows.append({
#                 "set": set_name,
#                 "run": run_i,
#                 "seed": seed_run,
#                 "n_rows": int(len(yr)),
#                 "n_features": int(len(feat_cols)),
#                 "n_cat": int(len(cat_in_use)),
#                 "auc": auc,
#                 "trees": int(model.tree_count_),
#                 "sec": float(wall),
#             })

#     df = pd.DataFrame(rows)
#     agg = (
#         df.groupby("set", as_index=False)
#           .agg(
#               auc_mean=("auc", "mean"),
#               auc_std=("auc", "std"),
#               sec_mean=("sec", "mean"),
#               n_features=("n_features", "first"),
#               n_cat=("n_cat", "first"),
#               n_rows=("n_rows", "first"),
#           )
#           .sort_values("auc_mean", ascending=False)
#           .reset_index(drop=True)
#     )

#     print("\n" + "=" * 90)
#     print("Summary (mean ± std)")
#     print("=" * 90)
#     display(agg)

#     return df

# # Main

# def step5_training_diagnostics(
#     *,
#     train: pd.DataFrame,
#     test: pd.DataFrame,
#     core: CFGCore,
#     prep: CFGPrep,
#     run_dir: Path,
#     cache_dir: Path,
#     features: List[str],
#     cat_cols: List[str],
#     num_cols: List[str],
#     feature_report: Optional[pd.DataFrame],
#     high_missing_thr: float = HIGH_MISSING_THR,
#     use_cache: bool = USE_CACHE,
#     sanitize_cfg: CatboostSanitizeCfg = SANITIZE,
#     holdout_cfg: HoldoutCfg = HOLDOUT,
#     cb_diag: CatBoostDiagParams = CB_DIAG,
# ) -> pd.DataFrame:
#     ID, TGT = core.id_col, core.target_col

#     if TGT not in train.columns:
#         raise ValueError("Step5: target missing in train")
#     if TGT in test.columns:
#         raise ValueError("Step5: target must not be in test")

#     bad = [c for c in features if c not in train.columns or c not in test.columns]
#     if bad:
#         raise ValueError(f"Step5: missing features in train/test (examples): {bad[:20]}")

#     # Make sure profession is derived-only (no raw text).
#     _assert_profession_features_only(features=features, prep=prep)

#     y = train[TGT].astype(np.int8).to_numpy()

#     feature_sets = build_feature_sets(
#         features_all=features,
#         cat_cols_all=cat_cols,
#         num_cols_all=num_cols,
#         train=train,
#         test=test,
#         feature_report=feature_report,
#         high_missing_thr=high_missing_thr,
#         prep=prep,
#     )

#     fp = {
#         "train_path": str(getattr(core, "train_path", "")),
#         "test_path": str(getattr(core, "test_path", "")),
#         "train_fp": _file_fingerprint(Path(getattr(core, "train_path"))) if hasattr(core, "train_path") else "",
#         "test_fp": _file_fingerprint(Path(getattr(core, "test_path"))) if hasattr(core, "test_path") else "",
#         "features_hash": _stable_hash_str("|".join(features), 16),
#         "cat_hash": _stable_hash_str("|".join(cat_cols), 12),
#         "num_hash": _stable_hash_str("|".join(num_cols), 12),
#         "high_missing_thr": float(high_missing_thr),
#         "sanitize_cfg": sanitize_cfg.__dict__,
#     }
#     cache_base = "step5_X_all_s_" + _stable_hash_str(json.dumps(fp, ensure_ascii=False, sort_keys=True), 16)

#     p_parq, p_meta = _cache_paths(cache_dir, cache_base)
#     if use_cache and p_parq.exists() and p_meta.exists():
#         X_all_s, y_cache, _ = _load_cached_matrix(cache_dir, cache_base)
#         if len(y_cache) != len(y) or (not np.array_equal(y_cache, y)):
#             raise ValueError("Step5: cached y differs; delete cache and re-run")
#         if list(X_all_s.columns) != list(features):
#             raise ValueError("Step5: cached X cols differ; delete cache and re-run")
#         print(f"[Step5] Loaded cache: {p_parq.name} | X={X_all_s.shape}")
#     else:
#         print("[Step5] Sanitizing full X...")
#         cfg = CatboostSanitizeCfg(
#             missing_cat_token=str(getattr(prep, "missing_token", sanitize_cfg.missing_cat_token)),
#             num_dtype=sanitize_cfg.num_dtype,
#         )
#         X_all_s = sanitize_catboost_df(train[features], cat_cols, cfg)

#         for c in cat_cols:
#             if c in X_all_s.columns and X_all_s[c].isna().any():
#                 raise ValueError(f"Step5: NA left in cat col: {c}")

#         meta = {
#             "created_at": dt.datetime.now().isoformat(timespec="seconds"),
#             "cache_base": cache_base,
#             "fingerprint": fp,
#             "X_shape": [int(X_all_s.shape[0]), int(X_all_s.shape[1])],
#         }
#         _save_cached_matrix(X_s=X_all_s, y=y, cache_dir=cache_dir, base=cache_base, meta=meta)
#         print(f"[Step5] Saved cache: {p_parq.name} | X={X_all_s.shape}")

#     df_runs = run_3x_holdout_suite(
#         X_all_s=X_all_s,
#         y=y,
#         feature_sets=feature_sets,
#         cat_cols_all=[c for c in cat_cols if c in features],
#         threads=int(core.threads),
#         holdout_cfg=holdout_cfg,
#         cb_params=cb_diag,
#     )

#     df_runs.to_csv(run_dir / "step5_holdout_runs.csv", index=False, encoding="utf-8")
#     _write_json(run_dir / "step5_feature_sets.json", feature_sets)
#     _write_json(
#         run_dir / "step5_cache_ref.json",
#         {"cache_base": cache_base, "parquet": str(p_parq), "meta": str(p_meta)},
#     )
#     _write_json(
#         run_dir / "step5_run_plan.json",
#         {"run_seeds": list(holdout_cfg.run_seeds), "sample_n": int(holdout_cfg.sample_n), "test_size": float(holdout_cfg.test_size)},
#     )

#     return df_runs


# # Run Step 5
# # If needed:
# # cat_cols, num_cols = _infer_cat_num(features, train)

# step5_runs = step5_training_diagnostics(
#     train=train,
#     test=test,
#     core=core,
#     prep=prep,
#     run_dir=RUN_DIR,
#     cache_dir=core.cache_dir,
#     features=features,
#     cat_cols=cat_cols,
#     num_cols=num_cols,
#     feature_report=feature_report,
# )

# step5_runs.head()


In [7]:
# 5 Training diagnostics (feature sets, 3x fixed runs, CatBoost).
# Reason behind CatBoost: convinient out of box, can handle many categories. LightGBM is a good option, but more difficult in setting from the beginning for me right now.

from __future__ import annotations

# Knobs (edit here)

from dataclasses import dataclass

@dataclass(frozen=True)
class CatBoostDiagParams:
    # quick sanity training, not final
    iterations: int = 400
    learning_rate: float = 0.07
    depth: int = 7
    l2_leaf_reg: float = 6.0
    random_strength: float = 1.0
    bagging_temperature: float = 0.5
    od_wait: int = 80

CB_DIAG = CatBoostDiagParams()

@dataclass(frozen=True)
class HoldoutCfg:
    # fixed 3 runs per set, same rows+split for all sets inside each run
    runs: int = 3
    run_seeds: tuple[int, int, int] = (10607, 20607, 30607)  # <- fixed
    sample_n: int = 200_000  # 0 -> full train
    test_size: float = 0.20
    seed: int = 42           # kept for cache fingerprint/back-compat, not used for runs
    verbose: int = 100

HOLDOUT = HoldoutCfg()

@dataclass(frozen=True)
class CatboostSanitizeCfg:
    # real NA/empty -> missing token
    missing_cat_token: str = "__MISSING__"
    num_dtype: str = "float32"

SANITIZE = CatboostSanitizeCfg()

HIGH_MISSING_THR: float = 0.95
USE_CACHE: bool = True


# Implementation

import json
import time
import hashlib
import datetime as dt
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


# IO

def _write_json(path: Path, obj: Any) -> None:
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def _read_json(path: Path) -> Dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


# Cache

def _stable_hash_str(s: str, n_bytes: int = 12) -> str:
    return hashlib.blake2b(s.encode("utf-8"), digest_size=n_bytes).hexdigest()


def _file_fingerprint(path: Path) -> str:
    st = path.stat()
    return f"{path.name}|size={st.st_size}|mtime_ns={st.st_mtime_ns}"


def _cache_paths(cache_dir: Path, base: str) -> Tuple[Path, Path]:
    return cache_dir / f"{base}.parquet", cache_dir / f"{base}.meta.json"


def _load_cached_matrix(cache_dir: Path, base: str) -> Tuple[pd.DataFrame, np.ndarray, Dict[str, Any]]:
    p_parq, p_meta = _cache_paths(cache_dir, base)
    df = pd.read_parquet(p_parq)
    meta = _read_json(p_meta)

    if "_target" not in df.columns:
        raise ValueError("Cache parquet is missing '_target' column")

    y = df["_target"].astype(np.int8).to_numpy()
    X = df.drop(columns=["_target"])
    return X, y, meta


def _save_cached_matrix(
    *,
    X_s: pd.DataFrame,
    y: np.ndarray,
    cache_dir: Path,
    base: str,
    meta: Dict[str, Any],
) -> None:
    p_parq, p_meta = _cache_paths(cache_dir, base)
    out = X_s.copy()
    out["_target"] = y.astype(np.int8)
    out.to_parquet(p_parq, index=False)
    _write_json(p_meta, meta)


# CatBoost sanitize (single source)

def sanitize_catboost_df(
    X: pd.DataFrame,
    cat_cols: Sequence[str],
    cfg: CatboostSanitizeCfg,
) -> pd.DataFrame:
    X2 = X.copy()
    cat_set = set(cat_cols)

    for c in X2.columns:
        if c in cat_set:
            s = X2[c].astype("string").str.strip()
            s = s.where(s.notna() & (s != ""), other=cfg.missing_cat_token)
            X2[c] = s.astype(str)
        else:
            X2[c] = pd.to_numeric(X2[c], errors="coerce").astype(cfg.num_dtype)

    return X2


# Feature sets

def _infer_cat_num(features: Sequence[str], train: pd.DataFrame) -> Tuple[List[str], List[str]]:
    feats = list(features)
    cat_cols = [c for c in feats if (pd.api.types.is_object_dtype(train[c]) or pd.api.types.is_string_dtype(train[c]))]
    num_cols = [c for c in feats if c not in cat_cols]
    return cat_cols, num_cols


def _ensure_missing_table(
    *,
    features_all: Sequence[str],
    train: pd.DataFrame,
    test: pd.DataFrame,
    feature_report: Optional[pd.DataFrame],
) -> pd.DataFrame:
    feats = list(features_all)
    for c in feats:
        if c not in train.columns or c not in test.columns:
            raise ValueError(f"Step5: feature missing in train/test: {c}")

    rows: Dict[str, Dict[str, float]] = {}

    if feature_report is not None:
        need = {"col", "train_missing", "test_missing"}
        if need.issubset(feature_report.columns):
            rep = feature_report[["col", "train_missing", "test_missing"]].copy()
            rep = rep.dropna(subset=["col"])
            for _, r in rep.iterrows():
                col = str(r["col"])
                if col:
                    rows[col] = {
                        "train_missing": float(r["train_missing"]),
                        "test_missing": float(r["test_missing"]),
                    }

    missing_cols = [c for c in feats if c not in rows]
    if missing_cols:
        tr_miss = train[missing_cols].isna().mean().astype(float)
        te_miss = test[missing_cols].isna().mean().astype(float)
        for c in missing_cols:
            rows[c] = {"train_missing": float(tr_miss[c]), "test_missing": float(te_miss[c])}

    miss = pd.DataFrame(
        [{"col": c, "train_missing": v["train_missing"], "test_missing": v["test_missing"]} for c, v in rows.items()]
    ).set_index("col")

    uncovered = sorted(set(feats) - set(miss.index))
    if uncovered:
        raise ValueError(f"Step5: missing table incomplete (examples): {uncovered[:20]}")

    return miss.loc[feats, ["train_missing", "test_missing"]]


def _profession_cols(prep: Any) -> List[str]:
    # Keep this list explicit and stable; drop only if the column exists.
    return [
        str(getattr(prep, "prof_col", "профессия_заемщика")),
        str(getattr(prep, "prof_superclass_col", "профессия_superclass")),
        str(getattr(prep, "prof_domain_col", "профессия_domain")),
        str(getattr(prep, "prof_seniority_col", "профессия_seniority")),
    ]


def build_feature_sets(
    *,
    features_all: Sequence[str],
    cat_cols_all: Sequence[str],
    num_cols_all: Sequence[str],
    train: pd.DataFrame,
    test: pd.DataFrame,
    feature_report: Optional[pd.DataFrame],
    high_missing_thr: float,
    prep: Any,
) -> Dict[str, List[str]]:
    feats_all = list(features_all)
    feat_set = set(feats_all)

    if not (0.0 <= float(high_missing_thr) <= 1.0):
        raise ValueError("high_missing_thr must be in [0, 1]")

    miss = _ensure_missing_table(
        features_all=feats_all,
        train=train,
        test=test,
        feature_report=feature_report,
    )

    drop_high = miss.index[
        (miss["train_missing"] > high_missing_thr) | (miss["test_missing"] > high_missing_thr)
    ].tolist()
    drop_high_set = set(drop_high)

    prof_cols = [c for c in _profession_cols(prep) if c in feat_set]
    prof_set = set(prof_cols)

    # Sets:
    # - All_wo_profession (drop raw profession + any derived if present)
    # - All_wo_profession_wo_high_missing (also drop high-missing > thr)
    # - Only_numerical
    # - Only_categorical (keep raw profession col)
    raw_prof = str(getattr(prep, "prof_col", "профессия_заемщика"))

    sets: Dict[str, List[str]] = {}
    
    sets["All_wo_profession"] = [c for c in feats_all if c not in prof_set]

    sets["All_wo_profession_wo_high_missing"] = [
        c for c in feats_all
        if (c not in prof_set) and (c not in drop_high_set)
    ]

    # Only_numerical: strictly numeric features.
    sets["Only_numerical"] = [c for c in num_cols_all if c in feat_set]

    # Only_categorical: strictly categorical features (profession stays here)
    sets["Only_categorical"] = [c for c in cat_cols_all if c in feat_set]

    # Basic checks.
    for name, cols in sets.items():
        if not cols:
            raise ValueError(f"Feature set '{name}' is empty")
        if len(set(cols)) != len(cols):
            raise ValueError(f"Feature set '{name}' has duplicates")
        bad = sorted(set(cols) - feat_set)
        if bad:
            raise ValueError(f"Feature set '{name}' has unknown cols (examples): {bad[:10]}")

    # Enforce stable order.
    order = [
        "All_wo_profession",
        "All_wo_profession_wo_high_missing",
        "Only_numerical",
        "Only_categorical",
    ]
    out: Dict[str, List[str]] = {}
    for k in order:
        out[k] = sets[k]

    # Diagnostics.
    if prof_cols:
        print("[Step5] profession cols present:", prof_cols)
    else:
        print("[Step5] profession cols present: none")

    print(f"[Step5] high-missing (> {high_missing_thr:g}) dropped count:", len(drop_high_set))

    return out


def _count_cat_num_for_set(
    feat_cols: Sequence[str],
    *,
    cat_cols_all: Sequence[str],
) -> Tuple[int, int]:
    cat_set = set(cat_cols_all)
    n_cat = sum(1 for c in feat_cols if c in cat_set)
    n_num = int(len(feat_cols) - n_cat)
    return n_num, n_cat


# -------------------------
# Holdout runs
# -------------------------
def _cb_params(
    *,
    seed: int,
    threads: int,
    verbose: int,
    p: CatBoostDiagParams,
) -> Dict[str, Any]:
    return dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=int(p.iterations),
        learning_rate=float(p.learning_rate),
        depth=int(p.depth),
        l2_leaf_reg=float(p.l2_leaf_reg),
        random_strength=float(p.random_strength),
        bagging_temperature=float(p.bagging_temperature),
        od_type="Iter",
        od_wait=int(p.od_wait),
        random_seed=int(seed),
        thread_count=int(threads) if threads is not None else -1,
        allow_writing_files=False,
        verbose=int(verbose),
    )


def _random_stratified_subset_idx(y: np.ndarray, n: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    y = np.asarray(y, dtype=np.int8)

    idx0 = np.flatnonzero(y == 0)
    idx1 = np.flatnonzero(y == 1)

    n = int(min(n, len(y)))
    if n <= 0:
        raise ValueError("sample_n must be positive")

    p1 = float(y.mean())
    n1 = int(round(n * p1))
    n0 = n - n1

    n0 = min(n0, len(idx0))
    n1 = min(n1, len(idx1))

    take0 = rng.choice(idx0, size=n0, replace=False) if n0 else np.array([], dtype=int)
    take1 = rng.choice(idx1, size=n1, replace=False) if n1 else np.array([], dtype=int)

    idx = np.concatenate([take0, take1])
    rng.shuffle(idx)
    return idx


def _make_run_plan(
    *,
    y: np.ndarray,
    holdout_cfg: HoldoutCfg,
) -> List[Dict[str, Any]]:
    # Same rows + same split for every feature set within a run.
    y = np.asarray(y, dtype=np.int8)

    if int(holdout_cfg.runs) != len(holdout_cfg.run_seeds):
        raise ValueError("HoldoutCfg: runs must match len(run_seeds)")

    plans: List[Dict[str, Any]] = []
    for run_i, seed_run in enumerate(holdout_cfg.run_seeds, 1):
        seed_run = int(seed_run)

        if holdout_cfg.sample_n and int(holdout_cfg.sample_n) > 0 and int(holdout_cfg.sample_n) < len(y):
            idx = _random_stratified_subset_idx(y, n=int(holdout_cfg.sample_n), seed=seed_run)
        else:
            idx = np.arange(len(y), dtype=int)

        yr = y[idx]
        tr_idx, va_idx = train_test_split(
            np.arange(len(yr), dtype=int),
            test_size=float(holdout_cfg.test_size),
            random_state=seed_run,
            stratify=yr,
        )
        plans.append({"run": run_i, "seed": seed_run, "idx": idx, "tr_idx": tr_idx, "va_idx": va_idx})

    return plans


def run_3x_holdout_suite(
    *,
    X_all_s: pd.DataFrame,
    y: np.ndarray,
    feature_sets: Dict[str, List[str]],
    cat_cols_all: List[str],
    threads: int,
    holdout_cfg: HoldoutCfg,
    cb_params: CatBoostDiagParams,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    y = np.asarray(y, dtype=np.int8)

    run_plan = _make_run_plan(y=y, holdout_cfg=holdout_cfg)
    print("[Step5] run seeds:", [p["seed"] for p in run_plan])

    for set_name, feat_cols in feature_sets.items():
        n_num, n_cat_total = _count_cat_num_for_set(feat_cols, cat_cols_all=cat_cols_all)

        print("\n" + "=" * 90)
        print(f"Feature set: {set_name} | n_feat={len(feat_cols)} | n_num={n_num} | n_cat={n_cat_total}")
        print("=" * 90)

        X_set = X_all_s[feat_cols]
        cat_in_use = [c for c in cat_cols_all if c in feat_cols]

        for p in run_plan:
            idx = p["idx"]
            tr_idx = p["tr_idx"]
            va_idx = p["va_idx"]
            seed_run = int(p["seed"])
            run_i = int(p["run"])

            Xr = X_set.iloc[idx].reset_index(drop=True)
            yr = y[idx]

            X_tr, X_va = Xr.iloc[tr_idx], Xr.iloc[va_idx]
            y_tr, y_va = yr[tr_idx], yr[va_idx]

            t0 = time.time()
            train_pool = Pool(X_tr, y_tr, cat_features=cat_in_use)
            valid_pool = Pool(X_va, y_va, cat_features=cat_in_use)

            model = CatBoostClassifier(**_cb_params(
                seed=seed_run,
                threads=threads,
                verbose=holdout_cfg.verbose,
                p=cb_params,
            ))
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

            proba = model.predict_proba(X_va)[:, 1]
            auc = float(roc_auc_score(y_va, proba))
            wall = time.time() - t0

            print(
                f"[{set_name}][run {run_i}/{holdout_cfg.runs}] ROC_AUC={auc:.6f} | "
                f"rows={len(yr)} | trees={int(model.tree_count_)} | sec={wall:.1f}"
            )

            rows.append({
                "set": set_name,
                "run": run_i,
                "seed": seed_run,
                "n_rows": int(len(yr)),
                "n_features": int(len(feat_cols)),
                "n_num": int(n_num),
                "n_cat": int(n_cat_total),
                "auc": auc,
                "trees": int(model.tree_count_),
                "sec": float(wall),
            })

    df = pd.DataFrame(rows)
    agg = (
        df.groupby("set", as_index=False)
          .agg(
              auc_mean=("auc", "mean"),
              auc_std=("auc", "std"),
              sec_mean=("sec", "mean"),
              n_features=("n_features", "first"),
              n_num=("n_num", "first"),
              n_cat=("n_cat", "first"),
              n_rows=("n_rows", "first"),
          )
          .sort_values("auc_mean", ascending=False)
          .reset_index(drop=True)
    )

    print("\n" + "=" * 90)
    print("Summary (mean ± std)")
    print("=" * 90)
    display(agg)

    return df


# -------------------------
# Main
# -------------------------
def step5_training_diagnostics(
    *,
    train: pd.DataFrame,
    test: pd.DataFrame,
    core: CFGCore,
    prep: CFGPrep,
    run_dir: Path,
    cache_dir: Path,
    features: List[str],
    cat_cols: List[str],
    num_cols: List[str],
    feature_report: Optional[pd.DataFrame],
    high_missing_thr: float = HIGH_MISSING_THR,
    use_cache: bool = USE_CACHE,
    sanitize_cfg: CatboostSanitizeCfg = SANITIZE,
    holdout_cfg: HoldoutCfg = HOLDOUT,
    cb_diag: CatBoostDiagParams = CB_DIAG,
) -> pd.DataFrame:
    ID, TGT = core.id_col, core.target_col

    if TGT not in train.columns:
        raise ValueError("Step5: target missing in train")
    if TGT in test.columns:
        raise ValueError("Step5: target must not be in test")

    bad = [c for c in features if c not in train.columns or c not in test.columns]
    if bad:
        raise ValueError(f"Step5: missing features in train/test (examples): {bad[:20]}")

    y = train[TGT].astype(np.int8).to_numpy()

    feature_sets = build_feature_sets(
        features_all=features,
        cat_cols_all=cat_cols,
        num_cols_all=num_cols,
        train=train,
        test=test,
        feature_report=feature_report,
        high_missing_thr=high_missing_thr,
        prep=prep,
    )

    fp = {
        "train_path": str(getattr(core, "train_path", "")),
        "test_path": str(getattr(core, "test_path", "")),
        "train_fp": _file_fingerprint(Path(getattr(core, "train_path"))) if hasattr(core, "train_path") else "",
        "test_fp": _file_fingerprint(Path(getattr(core, "test_path"))) if hasattr(core, "test_path") else "",
        "features_hash": _stable_hash_str("|".join(features), 16),
        "cat_hash": _stable_hash_str("|".join(cat_cols), 12),
        "num_hash": _stable_hash_str("|".join(num_cols), 12),
        "high_missing_thr": float(high_missing_thr),
        "sanitize_cfg": sanitize_cfg.__dict__,
    }
    cache_base = "step5_X_all_s_" + _stable_hash_str(json.dumps(fp, ensure_ascii=False, sort_keys=True), 16)

    p_parq, p_meta = _cache_paths(cache_dir, cache_base)
    if use_cache and p_parq.exists() and p_meta.exists():
        X_all_s, y_cache, _ = _load_cached_matrix(cache_dir, cache_base)
        if len(y_cache) != len(y) or (not np.array_equal(y_cache, y)):
            raise ValueError("Step5: cached y differs; delete cache and re-run")
        if list(X_all_s.columns) != list(features):
            raise ValueError("Step5: cached X cols differ; delete cache and re-run")
        print(f"[Step5] Loaded cache: {p_parq.name} | X={X_all_s.shape}")
    else:
        print("[Step5] Sanitizing full X...")
        cfg = CatboostSanitizeCfg(
            missing_cat_token=str(getattr(prep, "missing_token", sanitize_cfg.missing_cat_token)),
            num_dtype=sanitize_cfg.num_dtype,
        )
        X_all_s = sanitize_catboost_df(train[features], cat_cols, cfg)

        for c in cat_cols:
            if c in X_all_s.columns and X_all_s[c].isna().any():
                raise ValueError(f"Step5: NA left in cat col: {c}")

        meta = {
            "created_at": dt.datetime.now().isoformat(timespec="seconds"),
            "cache_base": cache_base,
            "fingerprint": fp,
            "X_shape": [int(X_all_s.shape[0]), int(X_all_s.shape[1])],
        }
        _save_cached_matrix(X_s=X_all_s, y=y, cache_dir=cache_dir, base=cache_base, meta=meta)
        print(f"[Step5] Saved cache: {p_parq.name} | X={X_all_s.shape}")

    df_runs = run_3x_holdout_suite(
        X_all_s=X_all_s,
        y=y,
        feature_sets=feature_sets,
        cat_cols_all=[c for c in cat_cols if c in features],
        threads=int(core.threads),
        holdout_cfg=holdout_cfg,
        cb_params=cb_diag,
    )

    df_runs.to_csv(run_dir / "step5_holdout_runs.csv", index=False, encoding="utf-8")
    _write_json(run_dir / "step5_feature_sets.json", feature_sets)
    _write_json(
        run_dir / "step5_cache_ref.json",
        {"cache_base": cache_base, "parquet": str(p_parq), "meta": str(p_meta)},
    )
    _write_json(
        run_dir / "step5_run_plan.json",
        {
            "run_seeds": list(holdout_cfg.run_seeds),
            "sample_n": int(holdout_cfg.sample_n),
            "test_size": float(holdout_cfg.test_size),
        },
    )

    return df_runs


# --- Run Step 5 ---
# If needed:
# cat_cols, num_cols = _infer_cat_num(features, train)

step5_runs = step5_training_diagnostics(
    train=train,
    test=test,
    core=core,
    prep=prep,
    run_dir=RUN_DIR,
    cache_dir=core.cache_dir,
    features=features,
    cat_cols=cat_cols,
    num_cols=num_cols,
    feature_report=feature_report,
)

step5_runs.head()


[Step5] profession cols present: ['профессия_заемщика']
[Step5] high-missing (> 0.95) dropped count: 4
[Step5] Loaded cache: step5_X_all_s_e6437c7152955c99a753f66de7d31fc6.parquet | X=(1210779, 105)
[Step5] run seeds: [10607, 20607, 30607]

Feature set: All_wo_profession | n_feat=104 | n_num=89 | n_cat=15
0:	test: 0.7037128	best: 0.7037128 (0)	total: 532ms	remaining: 3m 32s
100:	test: 0.7461218	best: 0.7461218 (100)	total: 37.4s	remaining: 1m 50s
200:	test: 0.7496145	best: 0.7496145 (200)	total: 1m 12s	remaining: 1m 11s
300:	test: 0.7511017	best: 0.7511125 (293)	total: 1m 47s	remaining: 35.3s
399:	test: 0.7520544	best: 0.7520544 (399)	total: 2m 22s	remaining: 0us

bestTest = 0.7520543663
bestIteration = 399

[All_wo_profession][run 1/3] ROC_AUC=0.752054 | rows=200000 | trees=400 | sec=146.5
0:	test: 0.7109171	best: 0.7109171 (0)	total: 409ms	remaining: 2m 43s
100:	test: 0.7488590	best: 0.7488590 (100)	total: 36s	remaining: 1m 46s
200:	test: 0.7520420	best: 0.7520420 (200)	total: 1m 11s

,set,auc_mean,auc_std,sec_mean,n_features,n_num,n_cat,n_rows
0,All_wo_profession,0.754854,0.002975,150.477450,104,89,15,200000
1,All_wo_profession_wo_high_missing,0.754467,0.002998,140.017711,100,86,14,200000
2,Only_numerical,0.749615,0.002934,27.909732,89,89,0,200000
3,Only_categorical,0.712164,0.004396,164.547701,16,0,16,200000


,set,run,seed,n_rows,n_features,n_num,n_cat,auc,trees,sec
0,All_wo_profession,1,10607,200000,104,89,15,0.752054,400,146.513311
1,All_wo_profession,2,20607,200000,104,89,15,0.754530,371,147.958652
2,All_wo_profession,3,30607,200000,104,89,15,0.757977,384,156.960386
3,All_wo_profession_wo_high_missing,1,10607,200000,100,86,14,0.751730,391,143.683930
4,All_wo_profession_wo_high_missing,2,20607,200000,100,86,14,0.754001,399,137.340796


In [71]:
# import json
# import pandas as pd

# # 1) feature_sets из артефакта Step5
# feature_sets = json.loads((RUN_DIR / "step5_feature_sets.json").read_text(encoding="utf-8"))

# # 2) X_all_s из кэша (паркет) по ссылке из step5_cache_ref.json
# cache_ref = json.loads((RUN_DIR / "step5_cache_ref.json").read_text(encoding="utf-8"))
# cache_base = cache_ref["cache_base"]  # имя базы кэша

# X_all_s, y_cache, meta = _load_cached_matrix(core.cache_dir, cache_base)
# print("Loaded X_all_s:", X_all_s.shape)

# # 3) печать: какие prof* колонки реально категориальные (dtype не numeric) в каждом feature set
# def print_prof_cats_by_X(X_all_s: pd.DataFrame, feature_sets: dict) -> None:
#     for set_name, cols in feature_sets.items():
#         cols_set = set(cols)
#         prof_cols = [c for c in X_all_s.columns if c in cols_set and c.lower().startswith("проф")]
#         prof_cat = [c for c in prof_cols if not pd.api.types.is_numeric_dtype(X_all_s[c])]
#         prof_num = [c for c in prof_cols if pd.api.types.is_numeric_dtype(X_all_s[c])]

#         print("=" * 80)
#         print(f"{set_name}: prof* total={len(prof_cols)} | cat={len(prof_cat)} | num={len(prof_num)}")
#         if prof_cat:
#             print("  cat:", prof_cat)
#         if prof_num:
#             print("  num:", prof_num)

# print_prof_cats_by_X(X_all_s, feature_sets)


Loaded X_all_s: (1210779, 107)
ALL: prof* total=3 | cat=3 | num=0
  cat: ['профессия_seniority', 'профессия_superclass', 'профессия_domain']
ALL_wo_high_missing>=0.95: prof* total=3 | cat=3 | num=0
  cat: ['профессия_seniority', 'профессия_superclass', 'профессия_domain']
NUM_ONLY: prof* total=0 | cat=0 | num=0
CAT_ONLY: prof* total=3 | cat=3 | num=0
  cat: ['профессия_seniority', 'профессия_superclass', 'профессия_domain']
ALL_wo_prof_domain: prof* total=2 | cat=2 | num=0
  cat: ['профессия_seniority', 'профессия_superclass']


In [74]:
# # ===== 5.extra) Run ALL_wo_high_missing_wo_prof_domain (robust: recovers y / X_all_s) =====
# import json
# import numpy as np
# import pandas as pd
# from catboost import CatBoostClassifier, Pool
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import roc_auc_score

# # --- 0) Recover y (target) ---
# TGT = core.target_col
# y_arr = train[TGT].astype(np.int8).to_numpy()

# # --- 1) Recover X_all_s (sanitized ALL matrix) if not in scope ---
# # Priority: already in memory -> load from step5_cache_ref.json -> fallback sanitize again
# if "X_all_s" not in globals():
#     cache_ref_path = RUN_DIR / "step5_cache_ref.json"
#     if cache_ref_path.exists():
#         ref = json.loads(cache_ref_path.read_text(encoding="utf-8"))
#         cache_base = ref["cache_base"]
#         X_all_s, y_cache, _ = _load_cached_matrix(core.cache_dir, cache_base)
#         # safety check
#         if len(y_cache) != len(y_arr) or (not np.array_equal(y_cache, y_arr)):
#             raise ValueError("extra: cached y differs from current train[TGT]; delete cache and re-run Step5")
#         print(f"[extra] loaded X_all_s from cache: {cache_base} | X={X_all_s.shape}")
#     else:
#         # last resort: sanitize again (slow, but works)
#         print("[extra] cache_ref not found; sanitizing X_all_s again (slow)...")
#         cfg = CatboostSanitizeCfg(
#             missing_cat_token=str(getattr(prep, "missing_token", SANITIZE.missing_cat_token)),
#             num_dtype=SANITIZE.num_dtype,
#         )
#         X_all_s = sanitize_catboost_df(train[features], cat_cols, cfg)
#         print(f"[extra] rebuilt X_all_s | X={X_all_s.shape}")

# # --- 2) Recover feature_sets if not in scope ---
# if "feature_sets" not in globals():
#     fs_path = RUN_DIR / "step5_feature_sets.json"
#     if not fs_path.exists():
#         raise ValueError("extra: feature_sets not found in memory and step5_feature_sets.json is missing")
#     feature_sets = json.loads(fs_path.read_text(encoding="utf-8"))
#     print(f"[extra] loaded feature_sets from: {fs_path.name} | keys={list(feature_sets.keys())}")

# # --- 3) Build new feature set: ALL_wo_high_missing>=thr minus profession_domain ---
# RUN_SEEDS = [10607, 20607, 30607]  # same as Step5 print
# BASE_SET = f"ALL_wo_high_missing>={HIGH_MISSING_THR:g}"
# assert BASE_SET in feature_sets, f"Missing base set: {BASE_SET}"

# PROF_DOMAIN_COL = str(getattr(prep, "prof_domain_col", "профессия_domain"))
# assert PROF_DOMAIN_COL in X_all_s.columns, f"{PROF_DOMAIN_COL} not found in X_all_s columns"

# new_name = "ALL_wo_high_missing_wo_prof_domain"
# new_feats = [c for c in feature_sets[BASE_SET] if c != PROF_DOMAIN_COL]
# print(f"[extra] {new_name}: n_feat={len(new_feats)} (dropped {PROF_DOMAIN_COL})")

# # --- 4) Cat features actually used (by dtype on sanitized X) ---
# def _cat_in_use(X: pd.DataFrame, feat_cols: list[str], cat_cols_all: list[str]) -> list[str]:
#     cols_set = set(feat_cols)
#     out = []
#     for c in cat_cols_all:
#         if c in cols_set and c in X.columns and not pd.api.types.is_numeric_dtype(X[c]):
#             out.append(c)
#     return out

# cat_in_use = _cat_in_use(X_all_s, new_feats, [c for c in cat_cols if c in X_all_s.columns])

# # --- 5) Run 3 fixed seeds (same sampling regime as Step5) ---
# rows = []
# X_set = X_all_s[new_feats]

# for run_i, seed_run in enumerate(RUN_SEEDS, 1):
#     # same stratified subsample logic as Step5
#     if HOLDOUT.sample_n and int(HOLDOUT.sample_n) > 0 and int(HOLDOUT.sample_n) < len(y_arr):
#         idx = _random_stratified_subset_idx(y_arr, n=int(HOLDOUT.sample_n), seed=int(seed_run))
#         Xr = X_set.iloc[idx].reset_index(drop=True)
#         yr = y_arr[idx]
#     else:
#         Xr = X_set
#         yr = y_arr

#     tr_idx, va_idx = train_test_split(
#         np.arange(len(yr)),
#         test_size=float(HOLDOUT.test_size),
#         random_state=int(seed_run),
#         stratify=yr,
#     )

#     X_tr, X_va = Xr.iloc[tr_idx], Xr.iloc[va_idx]
#     y_tr, y_va = yr[tr_idx], yr[va_idx]

#     train_pool = Pool(X_tr, y_tr, cat_features=cat_in_use)
#     valid_pool = Pool(X_va, y_va, cat_features=cat_in_use)

#     model = CatBoostClassifier(**_cb_params(
#         seed=int(seed_run),
#         threads=int(core.threads),
#         verbose=int(HOLDOUT.verbose),
#         p=CB_DIAG,
#     ))
#     model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

#     proba = model.predict_proba(X_va)[:, 1]
#     auc = float(roc_auc_score(y_va, proba))

#     print(f"[{new_name}][run {run_i}/3][seed={seed_run}] ROC_AUC={auc:.6f} | trees={int(model.tree_count_)} | rows={len(yr)}")

#     rows.append({
#         "set": new_name,
#         "run": run_i,
#         "seed": int(seed_run),
#         "n_rows": int(len(yr)),
#         "n_features": int(len(new_feats)),
#         "n_cat": int(len(cat_in_use)),
#         "auc": auc,
#         "trees": int(model.tree_count_),
#     })

# df_extra = pd.DataFrame(rows)
# display(df_extra)

# # save
# out_path = RUN_DIR / "step5_extra_all_wo_high_missing_wo_prof_domain.csv"
# df_extra.to_csv(out_path, index=False, encoding="utf-8")
# print("saved:", out_path)


[extra] ALL_wo_high_missing_wo_prof_domain: n_feat=102 (dropped профессия_domain)
0:	test: 0.6978908	best: 0.6978908 (0)	total: 373ms	remaining: 2m 28s
100:	test: 0.7470972	best: 0.7470972 (100)	total: 38.1s	remaining: 1m 52s
200:	test: 0.7511119	best: 0.7511119 (200)	total: 1m 15s	remaining: 1m 14s
300:	test: 0.7529934	best: 0.7529934 (300)	total: 1m 52s	remaining: 37s
399:	test: 0.7537407	best: 0.7537407 (399)	total: 2m 29s	remaining: 0us

bestTest = 0.7537407432
bestIteration = 399

[ALL_wo_high_missing_wo_prof_domain][run 1/3][seed=10607] ROC_AUC=0.753741 | trees=400 | rows=200000
0:	test: 0.7060523	best: 0.7060523 (0)	total: 410ms	remaining: 2m 43s
100:	test: 0.7504089	best: 0.7504089 (100)	total: 38.3s	remaining: 1m 53s
200:	test: 0.7534244	best: 0.7534244 (200)	total: 1m 15s	remaining: 1m 14s
300:	test: 0.7549017	best: 0.7549017 (300)	total: 1m 52s	remaining: 37.1s
399:	test: 0.7554727	best: 0.7554727 (399)	total: 2m 29s	remaining: 0us

bestTest = 0.7554727116
bestIteration = 39

,set,run,seed,n_rows,n_features,n_cat,auc,trees
0,ALL_wo_high_missing_wo_prof_domain,1,10607,200000,102,16,0.753741,400
1,ALL_wo_high_missing_wo_prof_domain,2,20607,200000,102,16,0.755473,400
2,ALL_wo_high_missing_wo_prof_domain,3,30607,200000,102,16,0.758061,398


saved: C:\Users\aa\Documents\SHIFT_ML\SHIFT_ML_2026_COMPETITION\outputs\runs\run_20260208_083226_f2a27c\step5_extra_all_wo_high_missing_wo_prof_domain.csv


In [ ]:
# # ===== 5.1) Quick CatBoost tuning on fixed subset+split =====
# import time
# import numpy as np
# import pandas as pd
# from dataclasses import dataclass
# from pathlib import Path
# from typing import Any, Dict, List, Tuple

# from catboost import CatBoostClassifier, Pool
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import roc_auc_score


# # ---------------------------------------------------------------------
# # Knobs
# # ---------------------------------------------------------------------
# @dataclass(frozen=True)
# class TuneCfg:
#     set_name: str = "ALL_wo_high_missing>=0.95"
#     sample_n: int = 200_000
#     subset_seed: int = 12345
#     split_seed: int = 54321
#     cb_seed: int = 777
#     out_csv: str = "step5_tune_quick.csv"
#     verbose: int = 100

# TUNE = TuneCfg()


# # ---------------------------------------------------------------------
# # Helpers
# # ---------------------------------------------------------------------
# def _get_threads(core: Any) -> int:
#     t = int(getattr(core, "threads", 8))
#     return t if t > 0 else -1


# def _build_pools_fixed(
#     *,
#     X_all_s: pd.DataFrame,
#     y: np.ndarray,
#     feat_cols: List[str],
#     cat_cols_all: List[str],
#     sample_n: int,
#     subset_seed: int,
#     split_seed: int,
#     test_size: float,
# ) -> Tuple[Pool, Pool, List[str], Dict[str, Any]]:
#     # Fixed rows.
#     idx = _random_stratified_subset_idx(y, n=int(sample_n), seed=int(subset_seed))
#     Xr = X_all_s[feat_cols].iloc[idx].reset_index(drop=True)
#     yr = y[idx]

#     # Fixed split.
#     tr_idx, va_idx = train_test_split(
#         np.arange(len(yr)),
#         test_size=float(test_size),
#         random_state=int(split_seed),
#         stratify=yr,
#     )
#     X_tr, X_va = Xr.iloc[tr_idx], Xr.iloc[va_idx]
#     y_tr, y_va = yr[tr_idx], yr[va_idx]

#     # CatBoost cat_features "as used".
#     cat_in_use = [
#         c for c in cat_cols_all
#         if c in feat_cols and c in X_tr.columns and not pd.api.types.is_numeric_dtype(X_tr[c])
#     ]

#     train_pool = Pool(X_tr, y_tr, cat_features=cat_in_use)
#     valid_pool = Pool(X_va, y_va, cat_features=cat_in_use)

#     meta = {
#         "n_rows": int(len(yr)),
#         "n_feat": int(len(feat_cols)),
#         "n_cat": int(len(cat_in_use)),
#         "test_size": float(test_size),
#         "subset_seed": int(subset_seed),
#         "split_seed": int(split_seed),
#     }
#     return train_pool, valid_pool, cat_in_use, meta


# def _base_params(*, core: Any, tune: TuneCfg) -> Dict[str, Any]:
#     return dict(
#         loss_function="Logloss",
#         eval_metric="AUC",
#         iterations=1200,
#         od_type="Iter",
#         od_wait=150,
#         metric_period=50,

#         random_seed=int(tune.cb_seed),
#         thread_count=_get_threads(core),
#         allow_writing_files=False,
#         verbose=int(tune.verbose),

#         # Baseline values
#         depth=7,
#         learning_rate=0.05,
#         l2_leaf_reg=8.0,
#         random_strength=1.0,
#         bagging_temperature=0.5,
#     )


# def _run_one(
#     *,
#     train_pool: Pool,
#     valid_pool: Pool,
#     y_valid: np.ndarray,
#     params: Dict[str, Any],
#     patch: Dict[str, Any],
# ) -> Dict[str, Any]:
#     p = dict(params)
#     p.update(patch)

#     t0 = time.time()
#     model = CatBoostClassifier(**p)
#     model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

#     proba = model.predict_proba(valid_pool)[:, 1]
#     auc = float(roc_auc_score(y_valid, proba))
#     sec = float(time.time() - t0)

#     out = dict(patch)
#     out.update({"auc": auc, "trees": int(model.tree_count_), "sec": sec})
#     print(f"[Tune] {patch} | AUC={auc:.6f} | trees={out['trees']} | sec={sec:.1f}")
#     return out


# # ---------------------------------------------------------------------
# # Run
# # ---------------------------------------------------------------------
# set_name = TUNE.set_name
# if set_name not in feature_sets:
#     raise ValueError(f"Tune: unknown feature set '{set_name}'. Available: {list(feature_sets)}")

# feat_cols = feature_sets[set_name]
# y_full = train[core.target_col].astype(np.int8).to_numpy()

# train_pool, valid_pool, cat_in_use, meta = _build_pools_fixed(
#     X_all_s=X_all_s,
#     y=y_full,
#     feat_cols=feat_cols,
#     cat_cols_all=[c for c in cat_cols if c in feat_cols],
#     sample_n=TUNE.sample_n,
#     subset_seed=TUNE.subset_seed,
#     split_seed=TUNE.split_seed,
#     test_size=HOLDOUT.test_size,
# )

# # y_valid explicitly for AUC.
# y_valid = valid_pool.get_label().astype(np.int8)

# print(f"[Tune] set={set_name} | rows={meta['n_rows']} | n_feat={meta['n_feat']} | n_cat={meta['n_cat']}")
# print(f"[Tune] threads={_get_threads(core)} | subset_seed={TUNE.subset_seed} | split_seed={TUNE.split_seed} | cb_seed={TUNE.cb_seed}")

# base = _base_params(core=core, tune=TUNE)

# # Stage A: coarse scan (9 runs).
# GRID_A = [
#     {"depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 6.0},
#     {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 8.0},
#     {"depth": 6, "learning_rate": 0.07, "l2_leaf_reg": 8.0},

#     {"depth": 7, "learning_rate": 0.05, "l2_leaf_reg": 6.0},
#     {"depth": 7, "learning_rate": 0.03, "l2_leaf_reg": 8.0},
#     {"depth": 7, "learning_rate": 0.07, "l2_leaf_reg": 8.0},

#     {"depth": 8, "learning_rate": 0.05, "l2_leaf_reg": 8.0},
#     {"depth": 8, "learning_rate": 0.03, "l2_leaf_reg": 10.0},
#     {"depth": 8, "learning_rate": 0.07, "l2_leaf_reg": 10.0},
# ]

# rows: List[Dict[str, Any]] = []
# for patch in GRID_A:
#     rows.append(_run_one(train_pool=train_pool, valid_pool=valid_pool, y_valid=y_valid, params=base, patch=patch))

# dfA = pd.DataFrame(rows).sort_values("auc", ascending=False).reset_index(drop=True)
# bestA = dfA.iloc[0].to_dict()
# print("\n[Tune] Stage A top-3")
# display(dfA.head(3))

# # Stage B: small tweaks around the winner (6 runs).
# best_patch = {k: bestA[k] for k in ["depth", "learning_rate", "l2_leaf_reg"]}

# GRID_B = [
#     {**best_patch, "bagging_temperature": 0.0},
#     {**best_patch, "bagging_temperature": 0.3},
#     {**best_patch, "bagging_temperature": 0.7},
#     {**best_patch, "bagging_temperature": 1.0},
#     {**best_patch, "l2_leaf_reg": float(best_patch["l2_leaf_reg"]) + 2.0},
#     {**best_patch, "learning_rate": max(0.02, float(best_patch["learning_rate"]) - 0.01)},
# ]

# rowsB: List[Dict[str, Any]] = []
# for patch in GRID_B:
#     rowsB.append(_run_one(train_pool=train_pool, valid_pool=valid_pool, y_valid=y_valid, params=base, patch=patch))

# dfB = pd.DataFrame(rowsB).sort_values("auc", ascending=False).reset_index(drop=True)
# print("\n[Tune] Stage B top-3")
# display(dfB.head(3))

# df_tune = (
#     pd.concat([dfA, dfB], ignore_index=True)
#       .sort_values("auc", ascending=False)
#       .reset_index(drop=True)
# )

# # Add meta columns for traceability.
# for k, v in meta.items():
#     df_tune[k] = v
# df_tune["set_name"] = set_name
# df_tune["threads"] = _get_threads(core)

# out_path = Path(RUN_DIR) / TUNE.out_csv
# df_tune.to_csv(out_path, index=False, encoding="utf-8")
# print("\n[Tune] Best configs:")
# display(df_tune.head(5))
# print("saved:", out_path)


[Tune] set=ALL_wo_high_missing>=0.95 | rows=200000 | n_feat=103 | n_cat=17
[Tune] threads=8 | subset_seed=12345 | split_seed=54321 | cb_seed=777


0:	test: 0.7108415	best: 0.7108415 (0)	total: 378ms	remaining: 7m 33s
100:	test: 0.7474015	best: 0.7474015 (100)	total: 34.4s	remaining: 6m 14s
200:	test: 0.7518047	best: 0.7518047 (200)	total: 1m 7s	remaining: 5m 37s
300:	test: 0.7539517	best: 0.7539517 (300)	total: 1m 39s	remaining: 4m 58s
400:	test: 0.7555495	best: 0.7555495 (400)	total: 2m 11s	remaining: 4m 22s
500:	test: 0.7564702	best: 0.7564702 (500)	total: 2m 43s	remaining: 3m 48s
600:	test: 0.7571781	best: 0.7571781 (600)	total: 3m 15s	remaining: 3m 14s
700:	test: 0.7577495	best: 0.7577495 (700)	total: 3m 47s	remaining: 2m 42s
800:	test: 0.7580432	best: 0.7580432 (800)	total: 4m 19s	remaining: 2m 9s
900:	test: 0.7584588	best: 0.7584588 (900)	total: 4m 52s	remaining: 1m 37s
1000:	test: 0.7587420	best: 0.7587420 (1000)	total: 5m 25s	remaining: 1m 4s
1100:	test: 0.7588939	best: 0.7588939 (1100)	total: 6m 4s	remaining: 32.8s
1199:	test: 0.7589534	best: 0.7589779 (1191)	total: 6m 36s	remaining: 0us

bestTest = 0.7589778625
bestIter

0:	test: 0.7108415	best: 0.7108415 (0)	total: 343ms	remaining: 6m 51s
100:	test: 0.7430899	best: 0.7430899 (100)	total: 31.9s	remaining: 5m 47s
200:	test: 0.7483404	best: 0.7483404 (200)	total: 1m 9s	remaining: 5m 46s
300:	test: 0.7509980	best: 0.7509980 (300)	total: 1m 42s	remaining: 5m 6s
400:	test: 0.7526061	best: 0.7526061 (400)	total: 2m 16s	remaining: 4m 32s
500:	test: 0.7539414	best: 0.7539414 (500)	total: 2m 49s	remaining: 3m 56s
600:	test: 0.7550486	best: 0.7550486 (600)	total: 3m 21s	remaining: 3m 21s
700:	test: 0.7558467	best: 0.7558467 (700)	total: 3m 59s	remaining: 2m 50s
800:	test: 0.7566037	best: 0.7566060 (798)	total: 4m 31s	remaining: 2m 15s
900:	test: 0.7571288	best: 0.7571293 (899)	total: 5m 6s	remaining: 1m 41s
1000:	test: 0.7574666	best: 0.7574778 (997)	total: 5m 42s	remaining: 1m 8s
1100:	test: 0.7578294	best: 0.7578380 (1098)	total: 6m 14s	remaining: 33.7s
1199:	test: 0.7580944	best: 0.7581025 (1198)	total: 6m 48s	remaining: 0us

bestTest = 0.7581025253
bestItera

0:	test: 0.7108415	best: 0.7108415 (0)	total: 338ms	remaining: 6m 45s
100:	test: 0.7498779	best: 0.7498779 (100)	total: 33s	remaining: 5m 58s
200:	test: 0.7535751	best: 0.7535751 (200)	total: 1m 9s	remaining: 5m 47s
300:	test: 0.7554708	best: 0.7554927 (297)	total: 1m 42s	remaining: 5m 5s
400:	test: 0.7565778	best: 0.7565778 (400)	total: 2m 14s	remaining: 4m 28s
500:	test: 0.7573728	best: 0.7573728 (500)	total: 2m 50s	remaining: 3m 58s
600:	test: 0.7577045	best: 0.7577352 (595)	total: 3m 24s	remaining: 3m 23s
700:	test: 0.7581360	best: 0.7581827 (680)	total: 4m	remaining: 2m 51s
800:	test: 0.7582377	best: 0.7582493 (798)	total: 4m 33s	remaining: 2m 16s
900:	test: 0.7582647	best: 0.7582839 (809)	total: 5m 10s	remaining: 1m 43s
1000:	test: 0.7584424	best: 0.7584519 (979)	total: 5m 43s	remaining: 1m 8s
1100:	test: 0.7581973	best: 0.7584519 (979)	total: 6m 19s	remaining: 34.2s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.7584518685
bestIteration = 979

Shrink model 

0:	test: 0.7179650	best: 0.7179650 (0)	total: 401ms	remaining: 8m
100:	test: 0.7485174	best: 0.7485174 (100)	total: 42.8s	remaining: 7m 45s
200:	test: 0.7525246	best: 0.7525246 (200)	total: 1m 22s	remaining: 6m 49s
300:	test: 0.7548008	best: 0.7548056 (299)	total: 2m 2s	remaining: 6m 6s
400:	test: 0.7561194	best: 0.7561194 (400)	total: 2m 43s	remaining: 5m 25s
500:	test: 0.7569581	best: 0.7569588 (499)	total: 3m 25s	remaining: 4m 46s
600:	test: 0.7577888	best: 0.7577893 (599)	total: 4m 6s	remaining: 4m 6s
700:	test: 0.7583770	best: 0.7583815 (697)	total: 4m 45s	remaining: 3m 23s
800:	test: 0.7586154	best: 0.7586154 (800)	total: 5m 32s	remaining: 2m 45s
900:	test: 0.7587679	best: 0.7587679 (900)	total: 6m 10s	remaining: 2m 3s
1000:	test: 0.7589661	best: 0.7589750 (994)	total: 6m 52s	remaining: 1m 22s
1100:	test: 0.7590941	best: 0.7590970 (1065)	total: 7m 31s	remaining: 40.6s
1199:	test: 0.7589665	best: 0.7590970 (1065)	total: 8m 13s	remaining: 0us

bestTest = 0.7590970028
bestIteration 

...

0:	test: 0.7240425	best: 0.7240425 (0)	total: 457ms	remaining: 9m 8s
100:	test: 0.7513116	best: 0.7513116 (100)	total: 51.9s	remaining: 9m 24s
200:	test: 0.7548489	best: 0.7548489 (200)	total: 1m 38s	remaining: 8m 11s
300:	test: 0.7565950	best: 0.7565950 (300)	total: 2m 27s	remaining: 7m 20s
400:	test: 0.7575307	best: 0.7575308 (397)	total: 3m 16s	remaining: 6m 30s
500:	test: 0.7582405	best: 0.7582978 (485)	total: 4m 5s	remaining: 5m 41s
600:	test: 0.7582882	best: 0.7584524 (544)	total: 4m 55s	remaining: 4m 54s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.7584523887
bestIteration = 544

Shrink model to first 545 iterations.
[Tune] {'depth': 8, 'learning_rate': 0.07, 'l2_leaf_reg': 10.0} | AUC=0.758452 | trees=545 | sec=339.8

[Tune] Stage A top-3


,depth,learning_rate,l2_leaf_reg,auc,trees,sec
0,8,0.05,8.0,0.759240,1030,584.389551
1,7,0.05,6.0,0.759097,1066,495.359794
2,8,0.03,10.0,0.759038,1191,584.055065


0:	test: 0.7240429	best: 0.7240429 (0)	total: 474ms	remaining: 9m 27s
100:	test: 0.7495403	best: 0.7495403 (100)	total: 49.8s	remaining: 9m 2s
200:	test: 0.7533542	best: 0.7533542 (200)	total: 1m 38s	remaining: 8m 10s
300:	test: 0.7554308	best: 0.7554308 (299)	total: 2m 29s	remaining: 7m 25s
400:	test: 0.7565962	best: 0.7565962 (400)	total: 3m 17s	remaining: 6m 33s
500:	test: 0.7576158	best: 0.7576276 (498)	total: 4m 8s	remaining: 5m 46s
600:	test: 0.7581626	best: 0.7582250 (595)	total: 4m 55s	remaining: 4m 54s
700:	test: 0.7587829	best: 0.7587858 (699)	total: 5m 46s	remaining: 4m 6s
800:	test: 0.7590295	best: 0.7591182 (773)	total: 6m 32s	remaining: 3m 15s
900:	test: 0.7589349	best: 0.7591199 (817)	total: 7m 21s	remaining: 2m 26s
1000:	test: 0.7591847	best: 0.7592162 (971)	total: 8m 11s	remaining: 1m 37s
1100:	test: 0.7589369	best: 0.7592403 (1029)	total: 8m 57s	remaining: 48.4s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.7592402708
bestIteration = 1029

Shrin

0:	test: 0.7240429	best: 0.7240429 (0)	total: 471ms	remaining: 9m 24s
100:	test: 0.7495403	best: 0.7495403 (100)	total: 47.5s	remaining: 8m 36s
200:	test: 0.7533542	best: 0.7533542 (200)	total: 1m 38s	remaining: 8m 8s
300:	test: 0.7554308	best: 0.7554308 (299)	total: 2m 24s	remaining: 7m 11s
400:	test: 0.7565962	best: 0.7565962 (400)	total: 3m 14s	remaining: 6m 28s
500:	test: 0.7576158	best: 0.7576276 (498)	total: 4m 5s	remaining: 5m 42s
600:	test: 0.7581626	best: 0.7582250 (595)	total: 4m 52s	remaining: 4m 51s
700:	test: 0.7587829	best: 0.7587858 (699)	total: 5m 41s	remaining: 4m 3s
800:	test: 0.7590295	best: 0.7591182 (773)	total: 6m 28s	remaining: 3m 13s
900:	test: 0.7589349	best: 0.7591199 (817)	total: 7m 21s	remaining: 2m 26s
1000:	test: 0.7591847	best: 0.7592162 (971)	total: 8m 9s	remaining: 1m 37s
1100:	test: 0.7589369	best: 0.7592403 (1029)	total: 9m	remaining: 48.6s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.7592402708
bestIteration = 1029

Shrink mod

0:	test: 0.7240429	best: 0.7240429 (0)	total: 462ms	remaining: 9m 14s


In [10]:
# 5.1 Quick tuning on Only_numerical (200k) + validate top-K on Step5 3x runs

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple
import time
import json
import datetime as dt

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


@dataclass(frozen=True)
class CatBoostTuneParams:
    
    iterations: int = 2000
    od_wait: int = 200
    metric_period: int = 50

    # defaults that get patched by grid
    depth: int = 7
    learning_rate: float = 0.05
    l2_leaf_reg: float = 8.0
    random_strength: float = 1.0

    # numeric-only: stable defaults
    bootstrap_type: str = "Bernoulli"
    subsample: float = 0.85
    rsm: float = 1.0
    min_data_in_leaf: int = 20


CB_TUNE = CatBoostTuneParams()


@dataclass(frozen=True)
class QuickTuneCfg:
    set_name: str = "Only_numerical"

    sample_n: int = 200_000
    subset_seed: int = 12345
    split_seed: int = 54321

    cb_seed: int = 777

    top_k: int = 3

    # output files
    out_quick_csv: str = "step5_1_tune_onlynum_quick.csv"
    out_valid_csv: str = "step5_1_tune_onlynum_validate3runs.csv"
    out_meta_json: str = "step5_1_tune_onlynum_meta.json"

    verbose: int = 200


TUNE_51 = QuickTuneCfg()

# Small utils

def _get_threads(core: Any) -> int:
    t = int(getattr(core, "threads", 8))
    return t if t > 0 else -1


def _write_json(path: Path, obj: Any) -> None:
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def _assert_onlynum_sane(X: pd.DataFrame, feat_cols: Sequence[str]) -> None:
    bad: List[str] = []
    for c in feat_cols:
        dtp = X[c].dtype
        if pd.api.types.is_object_dtype(dtp) or pd.api.types.is_string_dtype(dtp):
            bad.append(c)
    if bad:
        raise ValueError(f"Step5.1: Only_numerical contains non-numeric cols (examples): {bad[:15]}")

# CatBoost params builder

def _cb_params_tune_onlynum(
    *,
    seed: int,
    threads: int,
    verbose: int,
    p: CatBoostTuneParams,
) -> Dict[str, Any]:
    return dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=int(p.iterations),
        od_type="Iter",
        od_wait=int(p.od_wait),
        metric_period=int(p.metric_period),

        random_seed=int(seed),
        thread_count=int(threads) if threads is not None else -1,
        allow_writing_files=False,
        verbose=int(verbose),

        depth=int(p.depth),
        learning_rate=float(p.learning_rate),
        l2_leaf_reg=float(p.l2_leaf_reg),
        random_strength=float(p.random_strength),

        bootstrap_type=str(p.bootstrap_type),
        subsample=float(p.subsample),
        rsm=float(p.rsm),
        min_data_in_leaf=int(p.min_data_in_leaf),
    )


# Fixed subset + fixed split

def _build_fixed_pools_onlynum(
    *,
    X_all_s: pd.DataFrame,
    y: np.ndarray,
    feat_cols: Sequence[str],
    sample_n: int,
    subset_seed: int,
    split_seed: int,
    test_size: float,
) -> Tuple[Pool, Pool, np.ndarray, Dict[str, Any]]:
    idx = _random_stratified_subset_idx(y, n=int(sample_n), seed=int(subset_seed))

    Xr = X_all_s[list(feat_cols)].iloc[idx]
    yr = y[idx]

    tr_idx, va_idx = train_test_split(
        np.arange(len(yr), dtype=int),
        test_size=float(test_size),
        random_state=int(split_seed),
        stratify=yr,
    )

    X_tr, X_va = Xr.iloc[tr_idx], Xr.iloc[va_idx]
    y_tr, y_va = yr[tr_idx], yr[va_idx]

    train_pool = Pool(X_tr, y_tr)
    valid_pool = Pool(X_va, y_va)

    meta = {
        "n_rows": int(len(yr)),
        "n_feat": int(len(feat_cols)),
        "subset_seed": int(subset_seed),
        "split_seed": int(split_seed),
        "test_size": float(test_size),
    }
    return train_pool, valid_pool, y_va.astype(np.int8), meta


def _fit_eval_auc(
    *,
    train_pool: Pool,
    valid_pool: Pool,
    y_valid: np.ndarray,
    base_params: Dict[str, Any],
    patch: Dict[str, Any],
    tag: str,
) -> Dict[str, Any]:
    p = dict(base_params)
    p.update(patch)

    t0 = time.time()
    model = CatBoostClassifier(**p)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    proba = model.predict_proba(valid_pool)[:, 1]
    auc = float(roc_auc_score(y_valid, proba))
    sec = float(time.time() - t0)

    out = dict(patch)
    out.update({"auc": auc, "trees": int(model.tree_count_), "sec": sec})
    print(f"[{tag}] {patch} | AUC={auc:.6f} | trees={out['trees']} | sec={sec:.1f}")
    return out

# Validate top-K on Step5 3x run plan (same HOLDOUT seeds)

def _validate_topk_3runs_onlynum(
    *,
    X_all_s: pd.DataFrame,
    y: np.ndarray,
    feat_cols: Sequence[str],
    patches: List[Dict[str, Any]],
    core: Any,
    holdout_cfg: HoldoutCfg,
    tune_params: CatBoostTuneParams,
) -> pd.DataFrame:
    y = np.asarray(y, dtype=np.int8)

    run_plan = _make_run_plan(y=y, holdout_cfg=holdout_cfg)
    print("[Step5.1] validate run seeds:", [p["seed"] for p in run_plan])

    rows: List[Dict[str, Any]] = []
    threads = _get_threads(core)

    base = _cb_params_tune_onlynum(
        seed=777,  # overwritten per run
        threads=threads,
        verbose=int(holdout_cfg.verbose),
        p=tune_params,
    )

    for cand_i, patch in enumerate(patches, 1):
        print("\n" + "=" * 90)
        print(f"[Step5.1] candidate {cand_i}/{len(patches)}:", patch)
        print("=" * 90)

        for pl in run_plan:
            idx = pl["idx"]
            tr_idx = pl["tr_idx"]
            va_idx = pl["va_idx"]
            seed_run = int(pl["seed"])
            run_i = int(pl["run"])

            Xr = X_all_s[list(feat_cols)].iloc[idx]
            yr = y[idx]

            X_tr, X_va = Xr.iloc[tr_idx], Xr.iloc[va_idx]
            y_tr, y_va = yr[tr_idx], yr[va_idx]

            train_pool = Pool(X_tr, y_tr)
            valid_pool = Pool(X_va, y_va)

            p = dict(base)
            p.update(patch)
            p["random_seed"] = seed_run

            t0 = time.time()
            model = CatBoostClassifier(**p)
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

            proba = model.predict_proba(valid_pool)[:, 1]
            auc = float(roc_auc_score(y_va, proba))
            sec = float(time.time() - t0)

            print(f"[Step5.1][cand {cand_i}][run {run_i}] ROC_AUC={auc:.6f} | trees={int(model.tree_count_)} | sec={sec:.1f}")

            rows.append({
                "candidate": int(cand_i),
                "run": int(run_i),
                "seed": int(seed_run),
                "auc": float(auc),
                "trees": int(model.tree_count_),
                "sec": float(sec),
                **patch,
            })

    df = pd.DataFrame(rows)

    agg = (
        df.groupby("candidate", as_index=False)
          .agg(auc_mean=("auc", "mean"), auc_std=("auc", "std"), sec_mean=("sec", "mean"))
          .sort_values("auc_mean", ascending=False)
          .reset_index(drop=True)
    )

    print("\n" + "=" * 90)
    print("[Step5.1] validation summary (mean ± std)")
    print("=" * 90)
    display(agg)

    return df


# Main

def step5_1_quick_tune_only_numerical(
    *,
    train: pd.DataFrame,
    core: Any,                   
    X_all_s: pd.DataFrame,          
    feature_sets: Dict[str, List[str]],
    run_dir: Path,
    holdout_cfg: HoldoutCfg = HOLDOUT,
    tune_cfg: QuickTuneCfg = TUNE_51,
    tune_params: CatBoostTuneParams = CB_TUNE,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    set_name = str(tune_cfg.set_name)
    if set_name not in feature_sets:
        raise ValueError(f"Step5.1: unknown feature set '{set_name}'. Available: {list(feature_sets)}")

    feat_cols = feature_sets[set_name]
    y = train[core.target_col].astype(np.int8).to_numpy()

    _assert_onlynum_sane(X_all_s, feat_cols)

    threads = _get_threads(core)
    base = _cb_params_tune_onlynum(
        seed=int(tune_cfg.cb_seed),
        threads=threads,
        verbose=int(tune_cfg.verbose),
        p=tune_params,
    )

    train_pool, valid_pool, y_valid, meta = _build_fixed_pools_onlynum(
        X_all_s=X_all_s,
        y=y,
        feat_cols=feat_cols,
        sample_n=int(tune_cfg.sample_n),
        subset_seed=int(tune_cfg.subset_seed),
        split_seed=int(tune_cfg.split_seed),
        test_size=float(holdout_cfg.test_size),
    )

    print(f"[Step5.1] set={set_name} | rows={meta['n_rows']} | n_feat={meta['n_feat']} | test={meta['test_size']}")
    print(f"[Step5.1] threads={threads} | subset_seed={tune_cfg.subset_seed} | split_seed={tune_cfg.split_seed} | cb_seed={tune_cfg.cb_seed}")

    GRID_A = [
        {"depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 6.0},
        {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 10.0},
        {"depth": 6, "learning_rate": 0.07, "l2_leaf_reg": 10.0},

        {"depth": 7, "learning_rate": 0.05, "l2_leaf_reg": 6.0},
        {"depth": 7, "learning_rate": 0.03, "l2_leaf_reg": 10.0},
        {"depth": 7, "learning_rate": 0.07, "l2_leaf_reg": 10.0},

        {"depth": 8, "learning_rate": 0.05, "l2_leaf_reg": 10.0},
        {"depth": 8, "learning_rate": 0.03, "l2_leaf_reg": 14.0},
        {"depth": 8, "learning_rate": 0.07, "l2_leaf_reg": 14.0},
    ]

    rowsA: List[Dict[str, Any]] = []
    for patch in GRID_A:
        rowsA.append(_fit_eval_auc(
            train_pool=train_pool,
            valid_pool=valid_pool,
            y_valid=y_valid,
            base_params=base,
            patch=patch,
            tag="Step5.1-A",
        ))

    df_quick = pd.DataFrame(rowsA).sort_values("auc", ascending=False).reset_index(drop=True)

    print("\n[Step5.1] Stage A top-5")
    display(df_quick.head(5))

    # Traceability.
    df_quick["set_name"] = set_name
    df_quick["threads"] = int(threads)
    for k, v in meta.items():
        df_quick[k] = v

    out_quick = run_dir / str(tune_cfg.out_quick_csv)
    df_quick.to_csv(out_quick, index=False, encoding="utf-8")
    print(f"[Step5.1] quick results saved: {out_quick.name}")

    # Top-K patches for validation.
    top_k = int(max(1, tune_cfg.top_k))
    patches: List[Dict[str, Any]] = []
    for _, r in df_quick.head(top_k).iterrows():
        patch: Dict[str, Any] = {}
        for k in ["depth", "learning_rate", "l2_leaf_reg"]:
            patch[k] = r[k]
        patch["depth"] = int(patch["depth"])
        patch["learning_rate"] = float(patch["learning_rate"])
        patch["l2_leaf_reg"] = float(patch["l2_leaf_reg"])
        patches.append(patch)

    df_valid = _validate_topk_3runs_onlynum(
        X_all_s=X_all_s,
        y=y,
        feat_cols=feat_cols,
        patches=patches,
        core=core,
        holdout_cfg=holdout_cfg,
        tune_params=tune_params,
    )

    out_valid = run_dir / str(tune_cfg.out_valid_csv)
    df_valid.to_csv(out_valid, index=False, encoding="utf-8")
    print(f"[Step5.1] validation saved: {out_valid.name}")

    meta_out = {
        "created_at": dt.datetime.now().isoformat(timespec="seconds"),
        "set_name": set_name,
        "n_feat": int(len(feat_cols)),
        "tune_cfg": tune_cfg.__dict__,
        "tune_params": tune_params.__dict__,
        "holdout": {
            "runs": int(holdout_cfg.runs),
            "run_seeds": list(holdout_cfg.run_seeds),
            "sample_n": int(holdout_cfg.sample_n),
            "test_size": float(holdout_cfg.test_size),
        },
        "files": {
            "quick_csv": str(out_quick),
            "valid_csv": str(out_valid),
        },
        "top_patches": patches,
    }
    _write_json(run_dir / str(tune_cfg.out_meta_json), meta_out)

    return df_quick, df_valid



# Start
# If feature_sets is not in memory, uncomment:
feature_sets = _read_json(RUN_DIR / "step5_feature_sets.json")

# Step5.1 helper: load X_all_s from Step5 cache if not in memory
if "X_all_s" not in globals():
    cache_ref = _read_json(RUN_DIR / "step5_cache_ref.json")
    cache_base = str(cache_ref["cache_base"])
    X_all_s, y_cache, meta_cache = _load_cached_matrix(core.cache_dir, cache_base)
    print(f"[Step5.1] Loaded X_all_s from cache: {Path(cache_ref['parquet']).name} | X={X_all_s.shape}")

step5_1_quick, step5_1_valid = step5_1_quick_tune_only_numerical(
    train=train,
    core=core,
    X_all_s=X_all_s,
    feature_sets=feature_sets,
    run_dir=RUN_DIR,
    holdout_cfg=HOLDOUT,
    tune_cfg=TUNE_51,
    tune_params=CB_TUNE,
)

step5_1_quick.head(), step5_1_valid.head()


[Step5.1] set=Only_numerical | rows=200000 | n_feat=89 | test=0.2
[Step5.1] threads=8 | subset_seed=12345 | split_seed=54321 | cb_seed=777


0:	test: 0.6908149	best: 0.6908149 (0)	total: 70.7ms	remaining: 2m 21s
200:	test: 0.7447709	best: 0.7447709 (200)	total: 11.1s	remaining: 1m 39s
400:	test: 0.7499043	best: 0.7499043 (400)	total: 22.4s	remaining: 1m 29s
600:	test: 0.7520500	best: 0.7520745 (593)	total: 33.5s	remaining: 1m 18s
800:	test: 0.7530258	best: 0.7530258 (800)	total: 44.3s	remaining: 1m 6s
1000:	test: 0.7534308	best: 0.7534360 (999)	total: 55s	remaining: 54.9s
1200:	test: 0.7536028	best: 0.7536378 (1177)	total: 1m 5s	remaining: 43.7s
1400:	test: 0.7538206	best: 0.7539146 (1369)	total: 1m 16s	remaining: 32.8s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7539146481
bestIteration = 1369

Shrink model to first 1370 iterations.
[Step5.1-A] {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 6.0} | AUC=0.753915 | trees=1370 | sec=86.4


0:	test: 0.6908251	best: 0.6908251 (0)	total: 61.3ms	remaining: 2m 2s
200:	test: 0.7412997	best: 0.7412997 (200)	total: 11.5s	remaining: 1m 43s
400:	test: 0.7459041	best: 0.7459041 (400)	total: 22.8s	remaining: 1m 31s
600:	test: 0.7491709	best: 0.7491709 (600)	total: 34.2s	remaining: 1m 19s
800:	test: 0.7509363	best: 0.7509393 (799)	total: 45.4s	remaining: 1m 7s
1000:	test: 0.7520881	best: 0.7520881 (1000)	total: 56.9s	remaining: 56.7s
1200:	test: 0.7528657	best: 0.7528792 (1189)	total: 1m 7s	remaining: 44.9s
1400:	test: 0.7534941	best: 0.7534946 (1399)	total: 1m 19s	remaining: 33.8s
1600:	test: 0.7538682	best: 0.7538877 (1590)	total: 1m 32s	remaining: 23s
1800:	test: 0.7541701	best: 0.7541701 (1800)	total: 1m 43s	remaining: 11.4s
1999:	test: 0.7543215	best: 0.7543215 (1999)	total: 1m 54s	remaining: 0us

bestTest = 0.7543214822
bestIteration = 1999

[Step5.1-A] {'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 10.0} | AUC=0.754321 | trees=2000 | sec=114.7


0:	test: 0.6908251	best: 0.6908251 (0)	total: 63.4ms	remaining: 2m 6s
200:	test: 0.7474375	best: 0.7474375 (200)	total: 12.4s	remaining: 1m 51s
400:	test: 0.7519259	best: 0.7519259 (400)	total: 23.4s	remaining: 1m 33s
600:	test: 0.7533856	best: 0.7533984 (584)	total: 34.8s	remaining: 1m 20s
800:	test: 0.7540032	best: 0.7540195 (784)	total: 46.2s	remaining: 1m 9s
1000:	test: 0.7543281	best: 0.7543281 (1000)	total: 57.2s	remaining: 57.1s
1200:	test: 0.7543813	best: 0.7546499 (1116)	total: 1m 10s	remaining: 47.1s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7546498506
bestIteration = 1116

Shrink model to first 1117 iterations.
[Step5.1-A] {'depth': 6, 'learning_rate': 0.07, 'l2_leaf_reg': 10.0} | AUC=0.754650 | trees=1117 | sec=78.2


0:	test: 0.6945700	best: 0.6945700 (0)	total: 67.2ms	remaining: 2m 14s
200:	test: 0.7458845	best: 0.7458845 (200)	total: 12.6s	remaining: 1m 52s
400:	test: 0.7507841	best: 0.7507841 (400)	total: 27.3s	remaining: 1m 49s
600:	test: 0.7525143	best: 0.7525166 (599)	total: 39.9s	remaining: 1m 32s
800:	test: 0.7528122	best: 0.7528172 (793)	total: 54.7s	remaining: 1m 21s
1000:	test: 0.7531314	best: 0.7532660 (944)	total: 1m 8s	remaining: 1m 8s
1200:	test: 0.7531707	best: 0.7533004 (1186)	total: 1m 20s	remaining: 53.8s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7533003579
bestIteration = 1186

Shrink model to first 1187 iterations.
[Step5.1-A] {'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 6.0} | AUC=0.753300 | trees=1187 | sec=93.6


...

0:	test: 0.7047226	best: 0.7047226 (0)	total: 84.8ms	remaining: 2m 49s
100:	test: 0.7357665	best: 0.7357665 (100)	total: 8.11s	remaining: 2m 32s
200:	test: 0.7402983	best: 0.7402983 (200)	total: 15.9s	remaining: 2m 22s
300:	test: 0.7429086	best: 0.7429086 (300)	total: 25s	remaining: 2m 21s
400:	test: 0.7446313	best: 0.7446313 (400)	total: 34.5s	remaining: 2m 17s
500:	test: 0.7462288	best: 0.7462288 (500)	total: 42.5s	remaining: 2m 7s
600:	test: 0.7472617	best: 0.7472645 (598)	total: 50.1s	remaining: 1m 56s
700:	test: 0.7479376	best: 0.7479376 (700)	total: 58.9s	remaining: 1m 49s
800:	test: 0.7485050	best: 0.7485050 (800)	total: 1m 8s	remaining: 1m 42s
900:	test: 0.7489296	best: 0.7489296 (900)	total: 1m 18s	remaining: 1m 35s
1000:	test: 0.7494162	best: 0.7494162 (1000)	total: 1m 27s	remaining: 1m 26s
1100:	test: 0.7498239	best: 0.7498274 (1099)	total: 1m 34s	remaining: 1m 17s
1200:	test: 0.7499286	best: 0.7499286 (1200)	total: 1m 42s	remaining: 1m 7s
1300:	test: 0.7501466	best: 0.75014

0:	test: 0.6950134	best: 0.6950134 (0)	total: 79.7ms	remaining: 2m 39s
100:	test: 0.7399590	best: 0.7399590 (100)	total: 7.87s	remaining: 2m 28s
200:	test: 0.7448521	best: 0.7448521 (200)	total: 16.5s	remaining: 2m 27s
300:	test: 0.7476621	best: 0.7476621 (300)	total: 25.3s	remaining: 2m 22s
400:	test: 0.7493779	best: 0.7493779 (400)	total: 32.9s	remaining: 2m 11s
500:	test: 0.7510044	best: 0.7510203 (499)	total: 41.8s	remaining: 2m 5s
600:	test: 0.7518789	best: 0.7518815 (595)	total: 51.3s	remaining: 1m 59s
700:	test: 0.7526356	best: 0.7526368 (699)	total: 59.5s	remaining: 1m 50s
800:	test: 0.7531894	best: 0.7531894 (800)	total: 1m 7s	remaining: 1m 41s
900:	test: 0.7535980	best: 0.7535980 (900)	total: 1m 17s	remaining: 1m 34s
1000:	test: 0.7539220	best: 0.7539220 (1000)	total: 1m 25s	remaining: 1m 25s
1100:	test: 0.7542784	best: 0.7543089 (1089)	total: 1m 33s	remaining: 1m 16s
1200:	test: 0.7544542	best: 0.7544984 (1180)	total: 1m 42s	remaining: 1m 8s
1300:	test: 0.7547320	best: 0.754

,candidate,auc_mean,auc_std,sec_mean
0,2,0.751907,0.003230,141.567252
1,3,0.751553,0.002796,151.447051
2,1,0.750991,0.002370,80.560323


[Step5.1] validation saved: step5_1_tune_onlynum_validate3runs.csv


(   depth  learning_rate  l2_leaf_reg       auc  trees         sec  \
 0      6           0.07         10.0  0.754650   1117   78.152952   
 1      7           0.03         10.0  0.754401   1936  141.184301   
 2      8           0.03         14.0  0.754331   1433  142.846500   
 3      6           0.03         10.0  0.754321   2000  114.746546   
 4      6           0.05          6.0  0.753915   1370   86.443992   
 
          set_name  threads  n_rows  n_feat  subset_seed  split_seed  test_size  
 0  Only_numerical        8  200000      89        12345       54321        0.2  
 1  Only_numerical        8  200000      89        12345       54321        0.2  
 2  Only_numerical        8  200000      89        12345       54321        0.2  
 3  Only_numerical        8  200000      89        12345       54321        0.2  
 4  Only_numerical        8  200000      89        12345       54321        0.2  ,
    candidate  run   seed       auc  trees         sec  depth  learning_rate  \
 0   

In [11]:
best_candidate = int(
    step5_1_valid.groupby("candidate")["auc"].mean().sort_values(ascending=False).index[0]
)

best_row = (
    step5_1_quick.merge(
        step5_1_valid[step5_1_valid["candidate"] == best_candidate][["depth","learning_rate","l2_leaf_reg"]].drop_duplicates(),
        on=["depth","learning_rate","l2_leaf_reg"],
        how="inner",
    )
    .sort_values("auc", ascending=False)
    .iloc[0]
)

best_params = {
    "depth": int(best_row["depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "l2_leaf_reg": float(best_row["l2_leaf_reg"]),
}
_write_json(RUN_DIR / "step5_1_best_onlynum_params.json", best_params)
print("[Step5.1] best params saved:", best_params)


[Step5.1] best params saved: {'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 10.0}


In [14]:
# 6 Final training (All_wo_profession)

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple
import time
import json
import datetime as dt
import hashlib

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score



# Knobs

@dataclass(frozen=True)
class FinalTrainCfg:
    set_name: str = "All_wo_profession"

    # "outer" split for reporting AUC / early stopping
    valid_seed: int = 10607      
    valid_size: float = 0.20
    sample_n: int = 0            # 0 -> full train

    # upper cap for iterations; early stopping picks best_iteration
    iterations_cap: int = 500
    od_wait: int = 250

    # tuned core
    depth: int = 7
    learning_rate: float = 0.03
    l2_leaf_reg: float = 10.0

    # keep stable
    random_strength: float = 1.0
    bagging_temperature: float = 0.5

    # output
    model_name: str = "step6_cb_final_all_wo_profession.cbm"
    meta_name: str = "step6_final_meta.json"

    verbose: int = 200


FINAL6 = FinalTrainCfg()



# Small utils

def _get_threads(core: Any) -> int:
    t = int(getattr(core, "threads", 8))
    return t if t > 0 else -1


def _stable_hash_str(s: str, n_bytes: int = 12) -> str:
    return hashlib.blake2b(s.encode("utf-8"), digest_size=n_bytes).hexdigest()


def _load_feature_sets(run_dir: Path) -> Dict[str, List[str]]:
    p = run_dir / "step5_feature_sets.json"
    return _read_json(p)


def _load_X_all_s_from_step5_cache(*, core: Any, run_dir: Path) -> pd.DataFrame:
    cache_ref = _read_json(run_dir / "step5_cache_ref.json")
    cache_base = str(cache_ref["cache_base"])
    X_all_s, y_cache, meta_cache = _load_cached_matrix(core.cache_dir, cache_base)
    print(f"[Step6] Loaded X_all_s from Step5 cache: {Path(cache_ref['parquet']).name} | X={X_all_s.shape}")
    return X_all_s


def _pick_subset_idx(y: np.ndarray, n: int, seed: int) -> np.ndarray:
    if n and 0 < int(n) < len(y):
        return _random_stratified_subset_idx(y, n=int(n), seed=int(seed))
    return np.arange(len(y), dtype=int)


def _split_train_valid(
    y: np.ndarray,
    idx: np.ndarray,
    valid_size: float,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    yr = y[idx]
    tr, va = train_test_split(
        np.arange(len(yr), dtype=int),
        test_size=float(valid_size),
        random_state=int(seed),
        stratify=yr,
    )
    return tr, va


def _cb_params_final(
    *,
    core: Any,
    cfg: FinalTrainCfg,
    seed: int,
    iterations: int,
    verbose: int,
) -> Dict[str, Any]:
    return dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=int(iterations),

        learning_rate=float(cfg.learning_rate),
        depth=int(cfg.depth),
        l2_leaf_reg=float(cfg.l2_leaf_reg),

        random_strength=float(cfg.random_strength),
        bagging_temperature=float(cfg.bagging_temperature),

        od_type="Iter",
        od_wait=int(cfg.od_wait),

        random_seed=int(seed),
        thread_count=_get_threads(core),
        allow_writing_files=False,
        verbose=int(verbose),
    )

# Main

def step6_final_train(
    *,
    train: pd.DataFrame,
    test: pd.DataFrame,
    core: Any,                  
    prep: Any,                 
    run_dir: Path,
    features: List[str],
    cat_cols: List[str],
    cfg: FinalTrainCfg = FINAL6,
) -> CatBoostClassifier:
    ID, TGT = str(core.id_col), str(core.target_col)

    if TGT not in train.columns:
        raise ValueError("Step6: target missing in train")
    if TGT in test.columns:
        raise ValueError("Step6: target must not be in test")

    feature_sets = _load_feature_sets(run_dir)
    if cfg.set_name not in feature_sets:
        raise ValueError(f"Step6: unknown set '{cfg.set_name}'. Available: {list(feature_sets)}")
    feat_cols = list(feature_sets[cfg.set_name])
    print(f"[Step6] feature_set={cfg.set_name} | n_feat={len(feat_cols)}")

    # load sanitized train matrix from Step5 cache for consistency
    X_all_s = _load_X_all_s_from_step5_cache(core=core, run_dir=run_dir)
    y = train[TGT].astype(np.int8).to_numpy()

    # choose rows for AUC split
    idx = _pick_subset_idx(y, n=int(cfg.sample_n), seed=int(cfg.valid_seed))
    tr_idx, va_idx = _split_train_valid(y=y, idx=idx, valid_size=float(cfg.valid_size), seed=int(cfg.valid_seed))

    Xr = X_all_s[feat_cols].iloc[idx]
    yr = y[idx]

    X_tr, X_va = Xr.iloc[tr_idx], Xr.iloc[va_idx]
    y_tr, y_va = yr[tr_idx], yr[va_idx]

    cat_in_use = [c for c in cat_cols if c in feat_cols]
    print(f"[Step6] rows_for_fit={len(yr)} | train={len(y_tr)} | valid={len(y_va)} | cat_in_use={len(cat_in_use)}")
    if cat_in_use[:10]:
        print("[Step6] cat examples:", cat_in_use[:10])

    # A) Fit with holdout to get AUC + best_iteration
    params_a = _cb_params_final(
        core=core,
        cfg=cfg,
        seed=int(cfg.valid_seed),
        iterations=int(cfg.iterations_cap),
        verbose=int(cfg.verbose),
    )
    print("[Step6] stage A params:", {k: params_a[k] for k in ["iterations","learning_rate","depth","l2_leaf_reg","od_wait","random_seed"]})

    t0 = time.time()
    train_pool = Pool(X_tr, y_tr, cat_features=cat_in_use)
    valid_pool = Pool(X_va, y_va, cat_features=cat_in_use)

    m_a = CatBoostClassifier(**params_a)
    m_a.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    proba_va = m_a.predict_proba(valid_pool)[:, 1]
    auc_va = float(roc_auc_score(y_va, proba_va))
    best_iter = int(m_a.get_best_iteration() or (m_a.tree_count_ - 1))
    trees_a = int(m_a.tree_count_)
    sec_a = float(time.time() - t0)

    print(f"[Step6] stage A ROC_AUC={auc_va:.6f} | best_iter={best_iter} | trees={trees_a} | sec={sec_a:.1f}")

    # B) Refit on full train with fixed iterations=best_iter+1
    it_final = int(best_iter + 1)
    params_b = _cb_params_final(
        core=core,
        cfg=cfg,
        seed=42,
        iterations=it_final,
        verbose=max(50, int(cfg.verbose // 2)),
    )
    print("[Step6] stage B refit on full train:", {k: params_b[k] for k in ["iterations","learning_rate","depth","l2_leaf_reg","random_seed"]})

    t1 = time.time()
    full_pool = Pool(X_all_s[feat_cols], y, cat_features=cat_in_use)
    m_final = CatBoostClassifier(**params_b)
    m_final.fit(full_pool, use_best_model=False)
    sec_b = float(time.time() - t1)
    print(f"[Step6] stage B done | trees={int(m_final.tree_count_)} | sec={sec_b:.1f}")

    # Save model + meta (submission will be Step7)
    model_path = run_dir / cfg.model_name
    m_final.save_model(str(model_path))
    print("[Step6] saved model:", model_path)

    meta = {
        "created_at": dt.datetime.now().isoformat(timespec="seconds"),
        "feature_set": cfg.set_name,
        "n_feat": int(len(feat_cols)),
        "n_cat": int(len(cat_in_use)),
        "stageA": {
            "rows": int(len(yr)),
            "train_rows": int(len(y_tr)),
            "valid_rows": int(len(y_va)),
            "seed": int(cfg.valid_seed),
            "valid_size": float(cfg.valid_size),
            "auc": float(auc_va),
            "best_iter": int(best_iter),
            "trees": int(trees_a),
            "sec": float(sec_a),
            "params": params_a,
        },
        "stageB": {
            "train_rows": int(len(y)),
            "iterations": int(it_final),
            "sec": float(sec_b),
            "params": params_b,
        },
        "files": {
            "model": str(model_path),
        },
    }
    _write_json(run_dir / cfg.meta_name, meta)

    return m_final

# Start

cb_final = step6_final_train(
    train=train,
    test=test,
    core=core,
    prep=prep,
    run_dir=RUN_DIR,
    features=features,
    cat_cols=cat_cols,
    cfg=FINAL6,
)


[Step6] feature_set=All_wo_profession | n_feat=104
[Step6] Loaded X_all_s from Step5 cache: step5_X_all_s_e6437c7152955c99a753f66de7d31fc6.parquet | X=(1210779, 105)
[Step6] rows_for_fit=1210779 | train=968623 | valid=242156 | cat_in_use=15
[Step6] cat examples: ['срок_займа', 'рейтинг', 'допрейтинг', 'стаж', 'владение_жильем', 'подтвержден_ли_доход', 'цель_займа', 'регион', 'пос_стоп_фактор', 'первоначальный_статус_займа']
[Step6] stage A params: {'iterations': 500, 'learning_rate': 0.03, 'depth': 7, 'l2_leaf_reg': 10.0, 'od_wait': 250, 'random_seed': 10607}
0:	test: 0.7246062	best: 0.7246062 (0)	total: 1.87s	remaining: 15m 35s
200:	test: 0.7490817	best: 0.7490817 (200)	total: 6m 49s	remaining: 10m 9s
400:	test: 0.7530416	best: 0.7530416 (400)	total: 13m 37s	remaining: 3m 21s
499:	test: 0.7544540	best: 0.7544540 (499)	total: 16m 59s	remaining: 0us

bestTest = 0.7544539918
bestIteration = 499

[Step6] stage A ROC_AUC=0.754454 | best_iter=499 | trees=500 | sec=1038.0
[Step6] stage B ref

In [23]:
# 7A submisson.csv

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple
import time
import json
import datetime as dt
import hashlib

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

# Knobs

@dataclass(frozen=True)
class Step7_1ACfg:
    set_name: str = "All_wo_profession"
    model_name: str = "step6_cb_final_all_wo_profession.cbm"
    out_csv: str = "submisson.csv"
    id_header: str = "ID"
    proba_header: str = "Proba"


S71A = Step7_1ACfg()

# Helpers

def _stable_hash_str(s: str, n_bytes: int = 12) -> str:
    return hashlib.blake2b(s.encode("utf-8"), digest_size=n_bytes).hexdigest()


def _load_feature_sets(run_dir: Path) -> Dict[str, List[str]]:
    p = Path(run_dir) / "step5_feature_sets.json"
    if not p.exists():
        raise FileNotFoundError(f"Step7.1A: missing feature sets file: {p}")
    return _read_json(p)


def _load_model(run_dir: Path, model_name: str) -> CatBoostClassifier:
    p = Path(run_dir) / model_name
    if not p.exists():
        raise FileNotFoundError(f"Step7.1A: model not found: {p}")
    m = CatBoostClassifier()
    m.load_model(str(p))
    print(f"[Step7.1A] Loaded model: {p.name}")
    return m


def _sanitize_test_cached(
    *,
    test: pd.DataFrame,
    features: Sequence[str],
    cat_cols: Sequence[str],
    core: Any,
    prep: Any,
    sanitize_cfg: CatboostSanitizeCfg,
) -> pd.DataFrame:
    fp = {
        "test_path": str(getattr(core, "test_path", "")),
        "test_fp": _file_fingerprint(Path(getattr(core, "test_path"))) if hasattr(core, "test_path") else "",
        "features_hash": _stable_hash_str("|".join(list(features)), 16),
        "cat_hash": _stable_hash_str("|".join(list(cat_cols)), 12),
        "sanitize_cfg": sanitize_cfg.__dict__,
        "missing_token": str(getattr(prep, "missing_token", sanitize_cfg.missing_cat_token)),
    }
    cache_base = "step6_X_test_s_" + _stable_hash_str(json.dumps(fp, ensure_ascii=False, sort_keys=True), 16)
    p_parq, p_meta = _cache_paths(Path(core.cache_dir), cache_base)

    if p_parq.exists() and p_meta.exists():
        X_test_s = pd.read_parquet(p_parq)
        if list(X_test_s.columns) != list(features):
            raise ValueError("Step7.1A: cached test cols differ; delete step6_X_test_s_* cache and re-run")
        print(f"[Step7.1A] Loaded test sanitize cache: {p_parq.name} | X_test={X_test_s.shape}")
        return X_test_s

    print("[Step7.1A] Sanitizing test X (no cache found)...")
    cfg = CatboostSanitizeCfg(
        missing_cat_token=str(getattr(prep, "missing_token", sanitize_cfg.missing_cat_token)),
        num_dtype=sanitize_cfg.num_dtype,
    )
    X_test_s = sanitize_catboost_df(test[list(features)], cat_cols, cfg)

    for c in cat_cols:
        if c in X_test_s.columns and X_test_s[c].isna().any():
            raise ValueError(f"Step7.1A: NA left in test cat col: {c}")

    X_test_s.to_parquet(p_parq, index=False)
    _write_json(p_meta, {
        "created_at": dt.datetime.now().isoformat(timespec="seconds"),
        "cache_base": cache_base,
        "X_shape": [int(X_test_s.shape[0]), int(X_test_s.shape[1])],
        "fingerprint": fp,
    })
    print(f"[Step7.1A] Saved test sanitize cache: {p_parq.name} | X_test={X_test_s.shape}")
    return X_test_s


def step7_1a_predict_and_write_submisson(
    *,
    train: pd.DataFrame,
    test: pd.DataFrame,
    core: Any,
    prep: Any,
    run_dir: Path,
    features: List[str],
    cat_cols: List[str],
    cfg: Step7_1ACfg = S71A,
    sanitize_cfg: CatboostSanitizeCfg = SANITIZE,
) -> Path:
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    id_src = str(getattr(core, "id_col"))
    if id_src not in test.columns:
        raise ValueError(f"Step7.1A: core.id_col '{id_src}' missing in test")
    if cfg.id_header == cfg.proba_header:
        raise ValueError("Step7.1A: id_header and proba_header must differ")

    # load feature set
    feature_sets = _load_feature_sets(run_dir)
    if cfg.set_name not in feature_sets:
        raise ValueError(f"Step7.1A: unknown set '{cfg.set_name}'. Available: {list(feature_sets)}")
    feat_cols = list(feature_sets[cfg.set_name])
    print(f"[Step7.1A] feature_set={cfg.set_name} | n_feat={len(feat_cols)}")

    # load model
    model = _load_model(run_dir, cfg.model_name)

    # sanitize test (cached)
    X_test_s = _sanitize_test_cached(
        test=test,
        features=features,
        cat_cols=cat_cols,
        core=core,
        prep=prep,
        sanitize_cfg=sanitize_cfg,
    )
    X_te = X_test_s[feat_cols]

    # predict
    t0 = time.time()
    proba = model.predict_proba(X_te)[:, 1].astype(np.float32)
    sec = float(time.time() - t0)

    n = int(len(proba))
    n_nan = int(np.isnan(proba).sum())
    n_inf = int(np.isinf(proba).sum())
    print(f"[Step7.1A] predicted | n={n} | sec={sec:.1f} | nan={n_nan} | inf={n_inf}")
    print(f"[Step7.1A] proba[min/mean/max]={float(np.min(proba)):.6f}/{float(np.mean(proba)):.6f}/{float(np.max(proba)):.6f}")

    if n_nan or n_inf:
        raise ValueError("Step7.1A: proba contains NaN/Inf")

    # build output
    out = pd.DataFrame({
        cfg.id_header: test[id_src].values,
        cfg.proba_header: np.round(proba, 2),
    })

    dup = int(out[cfg.id_header].duplicated().sum())
    print(f"[Step7.1A] id duplicates: {dup}")
    if dup:
        raise ValueError(f"Step7.1A: duplicate IDs in output ({dup})")

    out_path = run_dir / cfg.out_csv
    out.to_csv(out_path, index=False, sep="\t", encoding="utf-8")
    print("[Step7.1A] saved:", out_path)

    return out_path

# Start
submisson_csv_path = step7_1a_predict_and_write_submisson(
    train=train,
    test=test,
    core=core,
    prep=prep,
    run_dir=RUN_DIR,
    features=features,
    cat_cols=cat_cols,
    cfg=S71A,
    sanitize_cfg=SANITIZE,
)

submisson_csv_path


[Step7.1A] feature_set=All_wo_profession | n_feat=104
[Step7.1A] Loaded model: step6_cb_final_all_wo_profession.cbm
[Step7.1A] Loaded test sanitize cache: step6_X_test_s_f9b1020d1a7d5cab601dc80413bc075a.parquet | X_test=(134531, 105)
[Step7.1A] predicted | n=134531 | sec=1.9 | nan=0 | inf=0
[Step7.1A] proba[min/mean/max]=0.010881/0.199477/0.930075
[Step7.1A] id duplicates: 0
[Step7.1A] saved: C:\Users\aa\Documents\SHIFT_ML\SHIFT_ML_2026_COMPETITION\outputs\runs\run_20260209_092450_05dcdc\submisson.csv


WindowsPath('C:/Users/aa/Documents/SHIFT_ML/SHIFT_ML_2026_COMPETITION/outputs/runs/run_20260209_092450_05dcdc/submisson.csv')

In [24]:
# 7B submisson.zip

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Union
import zipfile
import subprocess
import shutil


@dataclass(frozen=True)
class Step7_1BCfg:
    csv_name: str = "submisson.csv"
    req_name: str = "requirements.txt"
    nb_name: str = "competition.ipynb" 
    zip_name: str = "submisson.zip"
    notebook_src: Optional[str] = None


S71B = Step7_1BCfg()


def _write_requirements_txt(path: Path) -> None:
    # 1) try: current interpreter pip
    cmds = [
        [sys.executable, "-m", "pip", "freeze"],
        ["python", "-m", "pip", "freeze"],  # extra fallback if sys.executable is weird
    ]

    last_err: str | None = None
    for cmd in cmds:
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, check=False)
            out = (r.stdout or "").strip()
            err = (r.stderr or "").strip()

            print(f"[Step7.1B] requirements: cmd={' '.join(cmd)} | rc={r.returncode} | lines={0 if not out else out.count(chr(10))+1}")
            if err:
                print("[Step7.1B] requirements stderr (first 300 chars):", err[:300])

            if r.returncode == 0 and out:
                path.write_text(out + "\n", encoding="utf-8")
                return

            last_err = err or f"rc={r.returncode}, empty output"
        except Exception as e:
            last_err = repr(e)

    # 2) fallback: enumerate installed distributions
    print("[Step7.1B] requirements: falling back to importlib.metadata (pip freeze unavailable/empty)")
    dists = sorted(
        (f"{d.metadata['Name']}=={d.version}" for d in im.distributions() if d.metadata.get("Name")),
        key=str.lower,
    )
    if not dists:
        raise RuntimeError(f"Step7.1B: cannot build requirements.txt (no dists found). Last error: {last_err}")

    path.write_text("\n".join(dists) + "\n", encoding="utf-8")


def _find_notebook_src(*, preferred: Optional[str], name: str = "competition.ipynb") -> Path:
    if preferred:
        p = Path(preferred)
        if not p.exists():
            raise FileNotFoundError(f"Step7.1B: notebook_src not found: {p}")
        if p.suffix.lower() != ".ipynb":
            raise ValueError(f"Step7.1B: notebook_src must be .ipynb, got: {p.name}")
        return p

    p0 = Path(name)
    if p0.exists():
        return p0

    hits = [p for p in Path(".").rglob(name)]
    if not hits:
        raise FileNotFoundError(
            f"Step7.1B: cannot find '{name}'. Set S71B.notebook_src to the real path of your notebook."
        )

    hits.sort(key=lambda p: (len(p.as_posix()), p.as_posix()))
    return hits[0]


def step7_1b_pack_zip(
    *,
    run_dir: Path,
    cfg: Step7_1BCfg = S71B,
) -> Path:
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    # 1) check submisson.csv exists
    csv_path = run_dir / cfg.csv_name
    if not csv_path.exists():
        raise FileNotFoundError(f"Step7.1B: missing {cfg.csv_name}. Run Step7.1A first: {csv_path}")
    print("[Step7.1B] found:", csv_path)

    # 2) requirements.txt
    req_path = run_dir / cfg.req_name
    _write_requirements_txt(req_path)
    print("[Step7.1B] saved:", req_path)

    # 3) copy notebook into RUN_DIR (stable name)
    nb_src = _find_notebook_src(preferred=cfg.notebook_src, name=cfg.nb_name)
    nb_dst = run_dir / cfg.nb_name
    shutil.copy2(nb_src, nb_dst)
    print(f"[Step7.1B] copied notebook: {nb_src} -> {nb_dst}")

    # 4) zip (root files only)
    zip_path = run_dir / cfg.zip_name
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=cfg.csv_name)
        zf.write(req_path, arcname=cfg.req_name)
        zf.write(nb_dst, arcname=cfg.nb_name)

    # 5) quick verify zip content
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
    print("[Step7.1B] saved:", zip_path)
    print("[Step7.1B] zip contains:", names)

    must = {cfg.csv_name, cfg.req_name, cfg.nb_name}
    miss = sorted(must - set(names))
    if miss:
        raise ValueError(f"Step7.1B: zip missing files: {miss}")

    return zip_path

# Start

# If your notebook file is not named competition.ipynb or not in current folder,
# set: S71B = Step7_1BCfg(notebook_src="/path/to/your_notebook.ipynb")
submisson_zip_path = step7_1b_pack_zip(run_dir=RUN_DIR, cfg=S71B)
submisson_zip_path


[Step7.1B] found: C:\Users\aa\Documents\SHIFT_ML\SHIFT_ML_2026_COMPETITION\outputs\runs\run_20260209_092450_05dcdc\submisson.csv
[Step7.1B] requirements: cmd=C:\Users\aa\SHIFT_ML\Scripts\python.exe -m pip freeze | rc=0 | lines=117
[Step7.1B] saved: C:\Users\aa\Documents\SHIFT_ML\SHIFT_ML_2026_COMPETITION\outputs\runs\run_20260209_092450_05dcdc\requirements.txt
[Step7.1B] copied notebook: competition.ipynb -> C:\Users\aa\Documents\SHIFT_ML\SHIFT_ML_2026_COMPETITION\outputs\runs\run_20260209_092450_05dcdc\competition.ipynb
[Step7.1B] saved: C:\Users\aa\Documents\SHIFT_ML\SHIFT_ML_2026_COMPETITION\outputs\runs\run_20260209_092450_05dcdc\submisson.zip
[Step7.1B] zip contains: ['submisson.csv', 'requirements.txt', 'competition.ipynb']


WindowsPath('C:/Users/aa/Documents/SHIFT_ML/SHIFT_ML_2026_COMPETITION/outputs/runs/run_20260209_092450_05dcdc/submisson.zip')